# Data Fusion & Radiation Modelling (Eindhoven)

**Goal**

In this notebook, all preprocessed datasets are brought together on a common 0.05° SARAH grid to prepare inputs for solar radiation modelling.

**Dataset used**

- Sentinel-2 monthly surface features (NDVI, brightness, albedo, NIR)
- CAMS daily atmospheric variables (cloud fraction, AOD, water vapour)
- SARAH-3 daily GHI (ground truth)
- DEM terrain features (elevation, slope, aspect)

**This notebook covers:**

1. Loading all preprocessed data on the common 0.05° SARAH grid
2. Building a daily training dataset (features X, target y)
3. Training a baseline HGBR model
4. Training a CNN / U-Net model for full GHI maps
5. Preparing outputs that can be used in the final UI (GHI number, heatmap, simple explanations)


## 1. Imports & basic configuration

In [1]:
# Core
import os
from pathlib import Path
from datetime import datetime

# Numeric + data
import numpy as np
import pandas as pd
import xarray as xr

# Plots
import matplotlib.pyplot as plt

# For models later (we will fill these in when we reach modelling)
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score


## 2. Define data paths (all preprocessed files)

In [2]:
PROJECT_DIR = Path(r"C:\Users\Student\Desktop\SIS")

# SARAH-3 (ground truth)
SARAH_FILE = PROJECT_DIR / "data" / "SARAH3" / "sarah3_sis_eindhoven_0.05_2024-01-01_2025-04-01.nc"

# CAMS (atmospheric) 
CAMS_EAC4_FILE = PROJECT_DIR / "data" / "CAMS" / "cams_eac4_ehv_0.05_daily.nc"
CAMS_NRT_FILE  = PROJECT_DIR / "data" / "CAMS" / "cams_nrt_ehv_0.05_daily.nc"

# DEM (terrain) 
DEM_FILE = PROJECT_DIR / "data" / "DEM" / "dem_slope_aspect_on_sarah_0.05_ehv.nc"

# Sentinel-2 monthly composites
S2_MONTHLY_DIR = PROJECT_DIR / "data" / "S2_Features_Monthly"  

print("Project dir:", PROJECT_DIR)
print("SARAH file exists:", SARAH_FILE.exists())
print("CAMS EAC4 file exists:", CAMS_EAC4_FILE.exists())
print("CAMS NRT file exists:", CAMS_NRT_FILE.exists())
print("DEM file exists:", DEM_FILE.exists())
print("S2 monthly folder exists:", S2_MONTHLY_DIR.exists())

Project dir: C:\Users\Student\Desktop\SIS
SARAH file exists: True
CAMS EAC4 file exists: True
CAMS NRT file exists: True
DEM file exists: True
S2 monthly folder exists: True


## 3. Load SARAH-3 (reference grid & target)

SARAH defines:
- The spatial grid (20 × 20)
- The time axis
- The target variable (GHI / SIS)

In [3]:
sarah = xr.open_dataset(SARAH_FILE)
sarah

<xarray.Dataset> Size: 735kB
Dimensions:  (time: 457, lat: 20, lon: 20)
Coordinates:
  * time     (time) datetime64[ns] 4kB 2024-01-01 2024-01-02 ... 2025-04-01
  * lon      (lon) float32 80B 5.025 5.075 5.125 5.175 ... 5.875 5.925 5.975
  * lat      (lat) float32 80B 51.03 51.08 51.12 51.17 ... 51.88 51.92 51.97
Data variables:
    SIS      (time, lat, lon) float32 731kB ...
Attributes: (12/39)
    institution:                EUMETSAT/CMSAF
    Conventions:                CF-1.7,ACDD-1.3
    title:                      CM SAF Surface Solar Radiation Climate Data R...
    summary:                    This file contains data from the CM SAF Surfa...
    id:                         DOI:10.5676/EUM_SAF_CM/SARAH/V003
    product_version:            3.0
    ...                         ...
    platform:                   Earth Observation Satellites > METEOSAT > MET...
    CMSAF_repeat_cycles:        METEOSAT-10=48
    instrument_vocabulary:      GCMD Instruments, Version 8.6
    instrument:                 SEVIRI > Spinning Enhanced Visible and Infrar...
    variable_id:                SIS
    license:                    The CM SAF data are owned by EUMETSAT and are...

### Quick sanity check of SARAH

In [4]:
print("SARAH dims:", sarah.sizes)
print("Time range:",
      sarah.time.min().values, "→", sarah.time.max().values)
print("SIS range:",
      float(sarah.SIS.min()), "→", float(sarah.SIS.max()))

SARAH dims: Frozen({'time': 457, 'lat': 20, 'lon': 20})
Time range: 2024-01-01T00:00:00.000000000 → 2025-04-01T00:00:00.000000000
SIS range: 5.0 → 357.0


## 4. Load CAMS daily atmosphere (EAC4 + NRT)

CAMS files were already:
- Averaged to daily means
- Regridded to SARAH 0.05°
- Merged per product

In [5]:
cams_eac4 = xr.open_dataset(CAMS_EAC4_FILE)
cams_nrt  = xr.open_dataset(CAMS_NRT_FILE)

print("EAC4 dims:", cams_eac4.sizes)
print("NRT  dims:", cams_nrt.sizes)
print("CAMS variables:", list(cams_eac4.data_vars))


EAC4 dims: Frozen({'time': 366, 'lat': 20, 'lon': 20})
NRT  dims: Frozen({'time': 90, 'lat': 20, 'lon': 20})
CAMS variables: ['spatial_ref', 'tcc', 'aod550', 'tcwv']


### 4.1 Combine CAMS EAC4 + NRT into one dataset

We will:
- Concatenate along time
- Sort by time
- Clip to the same time range as SARAH.

In [6]:
# Concatenate + sort
cams_all = xr.concat([cams_eac4, cams_nrt], dim="time").sortby("time")

# Time sanity check (robust)
times = pd.to_datetime(cams_all["time"].values)

print("CAMS (raw concat) timesteps:", len(times))
print("CAMS (raw concat) time range:", times.min(), "→", times.max())
print("CAMS (raw concat) unique timestamps:", times.nunique())

# Drop duplicate timestamps if they exist
if times.nunique() != len(times):
    print("⚠ Duplicate CAMS timestamps found. Dropping duplicates...")
    cams_all = cams_all.sel(time=~cams_all.get_index("time").duplicated())
    cams_all = cams_all.sortby("time")

    times = pd.to_datetime(cams_all["time"].values)
    print("CAMS (deduped) timesteps:", len(times))
    print("CAMS (deduped) time range:", times.min(), "→", times.max())

# Align CAMS to SARAH time range
t_start = pd.to_datetime(sarah["time"].values[0])
t_end   = pd.to_datetime(sarah["time"].values[-1])

cams_all = cams_all.sel(time=slice(t_start, t_end)).sortby("time")

# Final check after alignment
times_aligned = pd.to_datetime(cams_all["time"].values)

print("\nAfter aligning to SARAH:")
print("CAMS aligned timesteps:", len(times_aligned))
print("CAMS aligned time range:", times_aligned.min(), "→", times_aligned.max())
print("CAMS aligned unique timestamps:", times_aligned.nunique())

# Check if CAMS covers every SARAH day (helps spot missing days)
sarah_days = pd.to_datetime(sarah["time"].values)
missing = pd.DatetimeIndex(sarah_days).difference(pd.DatetimeIndex(times_aligned))
print("Missing CAMS days vs SARAH:", len(missing))
if len(missing) > 0:
    print("First missing days:", list(missing[:10]))


CAMS (raw concat) timesteps: 456
CAMS (raw concat) time range: 2024-01-01 00:00:00 → 2025-03-31 00:00:00
CAMS (raw concat) unique timestamps: 456

After aligning to SARAH:
CAMS aligned timesteps: 456
CAMS aligned time range: 2024-01-01 00:00:00 → 2025-03-31 00:00:00
CAMS aligned unique timestamps: 456
Missing CAMS days vs SARAH: 1
First missing days: [Timestamp('2025-04-01 00:00:00')]


### 4.2 Check grid consistency: SARAH vs CAMS

In [7]:
print("SARAH lat/lon shape:", sarah.lat.shape, sarah.lon.shape)
print("CAMS  lat/lon shape:", cams_all.lat.shape, cams_all.lon.shape)

print("Lat equal? ", np.allclose(sarah.lat.values, cams_all.lat.values))
print("Lon equal? ", np.allclose(sarah.lon.values, cams_all.lon.values))

SARAH lat/lon shape: (20,) (20,)
CAMS  lat/lon shape: (20,) (20,)
Lat equal?  True
Lon equal?  True


## 5. Load DEM terrain on SARAH grid

From the DEM notebook we already regridded elevation, slope, aspect to 0.05°.
Now we just load that and verify it matches the same grid.

In [8]:
dem = xr.open_dataset(DEM_FILE)
dem

<xarray.Dataset> Size: 5kB
Dimensions:      (x: 20, y: 20)
Coordinates:
  * x            (x) float32 80B 5.025 5.075 5.125 5.175 ... 5.875 5.925 5.975
  * y            (y) float32 80B 51.03 51.08 51.12 51.17 ... 51.88 51.92 51.97
Data variables:
    spatial_ref  int64 8B ...
    elevation    (y, x) float32 2kB ...
    slope        (y, x) float32 2kB ...
    aspect       (y, x) float32 2kB ...

### Check DEM grid vs SARAH

In [9]:
# DEM uses x=lon, y=lat. SARAH uses lon/lat.
print("DEM lon/lat shape:", dem["x"].shape, dem["y"].shape)
print("SARAH lon/lat shape:", sarah["lon"].shape, sarah["lat"].shape)

print("Lon equal (DEM vs SARAH)?", np.allclose(dem["x"].values, sarah["lon"].values))
print("Lat equal (DEM vs SARAH)?", np.allclose(dem["y"].values, sarah["lat"].values))

DEM lon/lat shape: (20,) (20,)
SARAH lon/lat shape: (20,) (20,)
Lon equal (DEM vs SARAH)? True
Lat equal (DEM vs SARAH)? True


## 6. Preview of Sentinel-2 monthly composites 

The S2 monthly composites are still at 10 m resolution, which is much finer than 0.05°. For now, we only check if monthly GeoTIFFs exist and list them in order

In [10]:
s2_tifs = sorted(S2_MONTHLY_DIR.glob("S2_features_composite_*.tif"))
print("Number of monthly S2 composites found:", len(s2_tifs))
for p in s2_tifs:
    print(" -", p.name)


Number of monthly S2 composites found: 12
 - S2_features_composite_2024-01.tif
 - S2_features_composite_2024-02.tif
 - S2_features_composite_2024-03.tif
 - S2_features_composite_2024-04.tif
 - S2_features_composite_2024-05.tif
 - S2_features_composite_2024-06.tif
 - S2_features_composite_2024-07.tif
 - S2_features_composite_2024-08.tif
 - S2_features_composite_2024-09.tif
 - S2_features_composite_2024-10.tif
 - S2_features_composite_2024-11.tif
 - S2_features_composite_2024-12.tif


## 7. Resample Sentinel-2 monthly composites to the SARAH 0.05° grid

Here, we will:
- Read each monthly GeoTIFF (UTM 10 m)
- Reproject + resample to SARAH lat/lon grid (20×20)
- Store as an xarray Dataset with dims (month, lat, lon)

In [11]:
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds

# Output folder for S2 resampled products
S2_OUT_DIR = PROJECT_DIR / "data" / "S2_On_SARAH"
S2_OUT_DIR.mkdir(parents=True, exist_ok=True)

S2_SARAH_FILE = S2_OUT_DIR / "s2_monthly_features_on_sarah_0.05_2024.nc"

# SARAH grid info
sarah_lats = sarah["lat"].values
sarah_lons = sarah["lon"].values

# We construct a lat/lon grid extent from SARAH coordinates
lon_min, lon_max = float(sarah_lons.min()), float(sarah_lons.max())
lat_min, lat_max = float(sarah_lats.min()), float(sarah_lats.max())

# grid size (20x20)
out_h = sarah.sizes["lat"]
out_w = sarah.sizes["lon"]

# Make an affine transform for the output grid in EPSG:4326
# NOTE: from_bounds expects left, bottom, right, top (lon_min, lat_min, lon_max, lat_max)
dst_transform = from_bounds(lon_min, lat_min, lon_max, lat_max, out_w, out_h)
dst_crs = "EPSG:4326"

band_names = ["ndvi", "brightness", "albedo", "nir"]

def month_from_filename(p: Path):
    return p.stem.split("_")[-1]

def reproject_s2_month_to_sarah(tif_path: Path) -> xr.Dataset:
    month_str = month_from_filename(tif_path)

    with rasterio.open(tif_path) as src:
        src_crs = src.crs
        src_transform = src.transform
        src_nodata = src.nodata
        count = src.count

        # We'll store 4 bands -> (band, lat, lon)
        out_stack = np.full((count, out_h, out_w), np.nan, dtype="float32")

        for b in range(1, count + 1):
            src_data = src.read(b).astype("float32")

            dst_data = np.full((out_h, out_w), np.nan, dtype="float32")

            reproject(
                source=src_data,
                destination=dst_data,
                src_transform=src_transform,
                src_crs=src_crs,
                dst_transform=dst_transform,
                dst_crs=dst_crs,
                resampling=Resampling.average,  # average is good when aggregating 10m -> 0.05°
                src_nodata=src_nodata,
                dst_nodata=np.nan
            )

            out_stack[b - 1] = dst_data

    # Build xarray dataset
    ds = xr.Dataset(
        {
            band_names[i]: (("lat", "lon"), out_stack[i])
            for i in range(len(band_names))
        },
        coords={
            "lat": sarah_lats,
            "lon": sarah_lons,
            "month": month_str
        }
    ).expand_dims("month")

    return ds

print("Resampling Sentinel-2 monthly composites to SARAH grid...")
s2_monthly_tifs = sorted(S2_MONTHLY_DIR.glob("S2_features_composite_*.tif"))
print("Found monthly composites:", len(s2_monthly_tifs))

s2_monthly_on_sarah_list = []
for p in s2_monthly_tifs:
    ds_m = reproject_s2_month_to_sarah(p)
    s2_monthly_on_sarah_list.append(ds_m)

s2_monthly_on_sarah = xr.concat(s2_monthly_on_sarah_list, dim="month").sortby("month")
print(s2_monthly_on_sarah)

# Save
s2_monthly_on_sarah.to_netcdf(S2_SARAH_FILE)
print("✅ Saved:", S2_SARAH_FILE)


Resampling Sentinel-2 monthly composites to SARAH grid...
Found monthly composites: 12
<xarray.Dataset> Size: 77kB
Dimensions:     (month: 12, lat: 20, lon: 20)
Coordinates:
  * lat         (lat) float32 80B 51.03 51.08 51.12 51.17 ... 51.88 51.92 51.97
  * lon         (lon) float32 80B 5.025 5.075 5.125 5.175 ... 5.875 5.925 5.975
  * month       (month) <U7 336B '2024-01' '2024-02' ... '2024-11' '2024-12'
Data variables:
    ndvi        (month, lat, lon) float32 19kB 0.5906 0.6341 0.5808 ... nan nan
    brightness  (month, lat, lon) float32 19kB 0.05578 0.04629 ... nan nan
    albedo      (month, lat, lon) float32 19kB 0.2663 0.2717 0.2607 ... nan nan
    nir         (month, lat, lon) float32 19kB 0.1089 0.1083 0.107 ... nan nan
✅ Saved: C:\Users\Student\Desktop\SIS\data\S2_On_SARAH\s2_monthly_features_on_sarah_0.05_2024.nc


### Quick sanity check (S2 on SARAH grid)

In [12]:
print("S2 monthly on SARAH dims:", s2_monthly_on_sarah.sizes)
print("Months:", list(s2_monthly_on_sarah["month"].values))

for v in ["ndvi", "brightness", "albedo", "nir"]:
    arr = s2_monthly_on_sarah[v]
    print(f"{v:11s} range:",
          float(np.nanmin(arr.values)), "→", float(np.nanmax(arr.values)))

S2 monthly on SARAH dims: Frozen({'month': 12, 'lat': 20, 'lon': 20})
Months: [np.str_('2024-01'), np.str_('2024-02'), np.str_('2024-03'), np.str_('2024-04'), np.str_('2024-05'), np.str_('2024-06'), np.str_('2024-07'), np.str_('2024-08'), np.str_('2024-09'), np.str_('2024-10'), np.str_('2024-11'), np.str_('2024-12')]
ndvi        range: -0.09369616210460663 → 0.7984862923622131
brightness  range: 0.0 → 0.3298051953315735
albedo      range: 0.09211954474449158 → 1.0748000144958496
nir         range: 0.05640357360243797 → 1.0702250003814697


## 8. Convert Sentinel-2 from monthly to daily (align to SARAH time axis)

**Goal:** for each SARAH day, attach the month’s S2 features.

In [13]:
# Build month label per SARAH day (YYYY-MM)
sarah_time = pd.to_datetime(sarah["time"].values)
sarah_month_labels = sarah_time.strftime("%Y-%m")

# Make sure S2 months are easy to select
s2_month_index = pd.Index(s2_monthly_on_sarah["month"].values.astype(str))

# Create a daily list by picking the matching month
daily_s2_list = []
missing_months = []

for m in sarah_month_labels:
    if m not in s2_month_index:
        missing_months.append(m)
        # placeholder NaN grid if month missing
        empty = xr.Dataset(
            {k: (("lat", "lon"), np.full((out_h, out_w), np.nan, dtype="float32"))
             for k in band_names},
            coords={"lat": sarah_lats, "lon": sarah_lons}
        )
        daily_s2_list.append(empty)
    else:
        daily_s2_list.append(s2_monthly_on_sarah.sel(month=m).drop_vars("month"))

s2_daily = xr.concat(daily_s2_list, dim="time").assign_coords(time=sarah["time"].values)

print("S2 daily dims:", s2_daily.sizes)
print("Missing S2 months:", sorted(set(missing_months)))


S2 daily dims: Frozen({'time': 457, 'lat': 20, 'lon': 20})
Missing S2 months: ['2025-01', '2025-02', '2025-03', '2025-04']


## 9. Align everything to the same time axis (SARAH master)

We now create one fused dataset on the **SARAH 0.05° grid** with a shared daily time axis.

**Variables**
- Target: `SIS` (SARAH-3 daily GHI)
- CAMS (daily): `aod550`, `tcwv`, and `tcc` (only available in EAC4, not in NRT)
- DEM (static): `elevation`, `slope`, `aspect`
- Sentinel-2 (monthly → daily): `ndvi`, `brightness`, `albedo`, `nir` (available for 2024 only)

**Important note about missingness**
- CAMS NRT does not contain `tcc`, so `tcc` will be NaN from 2025-01-01 onward.
- Sentinel-2 monthly composites are currently only available for 2024 so S2 features will be NaN for 2025 days.
We keep these NaNs for transparency and handle them explicitly during modelling.


In [14]:
# Align CAMS to SARAH time 
cams_all = cams_all.sortby("time")
cams_all = cams_all.sel(time=slice(sarah.time.values[0], sarah.time.values[-1]))
cams_all = cams_all.reindex(time=sarah["time"].values)

# Verify CAMS grid matches SARAH grid 
assert np.allclose(cams_all["lat"].values, sarah["lat"].values), "CAMS lat grid != SARAH lat grid"
assert np.allclose(cams_all["lon"].values, sarah["lon"].values), "CAMS lon grid != SARAH lon grid"

# DEM: rename (x,y) -> (lon,lat) and broadcast to time
dem_ll = dem.rename({"x": "lon", "y": "lat"})
assert np.allclose(dem_ll["lat"].values, sarah["lat"].values), "DEM lat grid != SARAH lat grid"
assert np.allclose(dem_ll["lon"].values, sarah["lon"].values), "DEM lon grid != SARAH lon grid"

dem_time = dem_ll[["elevation", "slope", "aspect"]].expand_dims(time=sarah["time"].values)

# Build fused dataset 
def pick(ds, name):
    return ds[name] if (ds is not None and name in ds.data_vars) else None

fused_vars = {
    "SIS": sarah["SIS"],

    # CAMS (always present)
    "aod550": pick(cams_all, "aod550"),
    "tcwv":   pick(cams_all, "tcwv"),

    # DEM (static)
    "elevation": dem_time["elevation"],
    "slope":     dem_time["slope"],
    "aspect":    dem_time["aspect"],

    # S2 daily
    "ndvi":       s2_daily["ndvi"],
    "brightness": s2_daily["brightness"],
    "albedo":     s2_daily["albedo"],
    "nir":        s2_daily["nir"],
}

# tcc is missing in CAMS NRT but may exist in the combined dataset (NaNs during NRT period)
if "tcc" in cams_all.data_vars:
    fused_vars["tcc"] = cams_all["tcc"]

# remove None entries (if a variable doesn't exist)
fused_vars = {k: v for k, v in fused_vars.items() if v is not None}

fused = xr.Dataset(fused_vars)

print(fused)
print("\n✅ All datasets aligned:", fused.sizes)


<xarray.Dataset> Size: 8MB
Dimensions:     (time: 457, lon: 20, lat: 20)
Coordinates:
  * time        (time) datetime64[ns] 4kB 2024-01-01 2024-01-02 ... 2025-04-01
  * lon         (lon) float32 80B 5.025 5.075 5.125 5.175 ... 5.875 5.925 5.975
  * lat         (lat) float32 80B 51.03 51.08 51.12 51.17 ... 51.88 51.92 51.97
Data variables:
    SIS         (time, lat, lon) float32 731kB 34.0 35.0 34.0 ... 211.0 207.0
    aod550      (time, lat, lon) float32 731kB 0.1575 0.1575 0.1575 ... nan nan
    tcwv        (time, lat, lon) float32 731kB 12.48 12.48 12.48 ... nan nan nan
    elevation   (time, lat, lon) float32 731kB 18.92 20.24 27.68 ... 13.88 12.12
    slope       (time, lat, lon) float32 731kB 4.761 1.763 1.947 ... 3.061 1.051
    aspect      (time, lat, lon) float32 731kB 314.5 197.7 249.9 ... 199.5 252.0
    ndvi        (time, lat, lon) float32 731kB 0.5906 0.6341 0.5808 ... nan nan
    brightness  (time, lat, lon) float32 731kB 0.05578 0.04629 ... nan nan
    albedo      (time,

### Quick check: missing values per variable

In [15]:
def nan_ratio_total(da):
    vals = da.values
    return float(np.isnan(vals).sum()) / vals.size

def nan_ratio_per_day(da):
    return np.isnan(da.values).mean(axis=(1, 2))

print("\nOverall NaN ratios:")
for v in fused.data_vars:
    if "time" in fused[v].dims:
        print(f"{v:12s} NaN ratio:", round(nan_ratio_total(fused[v]), 4))

# tcc missing days (expected: NRT period + any inserted missing day)
if "tcc" in fused.data_vars:
    tcc_day = nan_ratio_per_day(fused["tcc"])
    bad = pd.to_datetime(fused.time.values)[tcc_day == 1.0]
    print("\n`tcc` days fully missing:", int((tcc_day == 1.0).sum()), "/", len(tcc_day))
    if len(bad) > 0:
        print("First missing `tcc` days:", list(bad[:5]))
        print("Last missing `tcc` days :", list(bad[-5:]))



Overall NaN ratios:
SIS          NaN ratio: 0.0
aod550       NaN ratio: 0.0022
tcwv         NaN ratio: 0.0022
elevation    NaN ratio: 0.0
slope        NaN ratio: 0.0
aspect       NaN ratio: 0.0
ndvi         NaN ratio: 0.4481
brightness   NaN ratio: 0.4454
albedo       NaN ratio: 0.4481
nir          NaN ratio: 0.4481
tcc          NaN ratio: 0.1991

`tcc` days fully missing: 91 / 457
First missing `tcc` days: [Timestamp('2025-01-01 00:00:00'), Timestamp('2025-01-02 00:00:00'), Timestamp('2025-01-03 00:00:00'), Timestamp('2025-01-04 00:00:00'), Timestamp('2025-01-05 00:00:00')]
Last missing `tcc` days : [Timestamp('2025-03-28 00:00:00'), Timestamp('2025-03-29 00:00:00'), Timestamp('2025-03-30 00:00:00'), Timestamp('2025-03-31 00:00:00'), Timestamp('2025-04-01 00:00:00')]


## 10. Build the daily training dataset (X, y)

We convert the fused xarray dataset into a tabular format:

- Each row = one grid cell on one day
- Features (X) come from: CAMS + DEM + Sentinel-2 + time features
- Target (y) = SARAH-3 `SIS` (GHI)

We also create:
- Training split: 2024 (full feature availability)
- Validation split: 2025-01 to 2025-03 (reduced feature availability)


In [16]:
# Convert to DataFrame (time, lat, lon -> rows)
df = fused.to_dataframe().reset_index()

# Basic sanity
print("Raw rows:", len(df))
print("Columns:", df.columns.tolist())
print("Time range:", df["time"].min(), "→", df["time"].max())

# Add time features (seasonality)
df["doy"] = df["time"].dt.dayofyear.astype(int)
df["doy_sin"] = np.sin(2 * np.pi * df["doy"] / 365.25)
df["doy_cos"] = np.cos(2 * np.pi * df["doy"] / 365.25)

# Add lat/lon as numeric features (often helps non-spatial models)
df["lat_f"] = df["lat"].astype(float)
df["lon_f"] = df["lon"].astype(float)

# Solar geometry: extraterrestrial radiation (I0) and clearness index target (Kt)

def extraterrestrial_radiation_Wm2(lat_deg, doy):
    lat = np.deg2rad(lat_deg.astype(float))
    doy = doy.astype(float)

    dr = 1 + 0.033 * np.cos(2 * np.pi * doy / 365.0)
    delta = 0.409 * np.sin(2 * np.pi * doy / 365.0 - 1.39)
    ws = np.arccos(np.clip(-np.tan(lat) * np.tan(delta), -1, 1))

    Gsc = 1367.0  # solar constant (W/m²)

    Ra = (24 * 3600 / np.pi) * Gsc * dr * (
        ws * np.sin(lat) * np.sin(delta) +
        np.cos(lat) * np.cos(delta) * np.sin(ws)
    )

    return Ra / (24 * 3600)  # daily mean W/m²


# Compute extraterrestrial radiation per pixel per day
df["I0"] = extraterrestrial_radiation_Wm2(df["lat_f"], df["doy"])

# Clearness index (dimensionless target)
df["Kt"] = df["SIS"] / df["I0"]

# Physically reasonable bounds
df["Kt"] = df["Kt"].clip(0.0, 1.2)

print("I0 range:", df["I0"].min(), "→", df["I0"].max())
print("Kt range:", df["Kt"].min(), "→", df["Kt"].max())

# Define feature sets
FULL_FEATURES = [
    # CAMS
    "tcc", "aod550", "tcwv",
    # DEM
    "elevation", "slope", "aspect",
    # S2
    "ndvi", "brightness", "albedo", "nir",
    # time + location
    "doy_sin", "doy_cos", "lat_f", "lon_f"
]

REDUCED_FEATURES = [
    # CAMS (variables that exist in both EAC4 + NRT)
    "aod550", "tcwv",
    # DEM
    "elevation", "slope", "aspect",
    # time + location
    "doy_sin", "doy_cos", "lat_f", "lon_f"
]

TARGET = "Kt"

# Split by time
train_mask = (df["time"] >= "2024-01-01") & (df["time"] <= "2024-12-31")
val_mask   = (df["time"] >= "2025-01-01") & (df["time"] <= "2025-03-31")

df_train = df.loc[train_mask].copy()
df_val   = df.loc[val_mask].copy()

print("\nTrain days:", df_train["time"].nunique(), " rows:", len(df_train))
print("Val days  :", df_val["time"].nunique(), " rows:", len(df_val))

# Drop rows with missing target 
df_train = df_train.dropna(subset=[TARGET])
df_val   = df_val.dropna(subset=[TARGET])

# Build X/y for FULL model (2024)
train_full = df_train.dropna(subset=FULL_FEATURES).copy()
X_train_full = train_full[FULL_FEATURES]
y_train_full = train_full[TARGET].astype(float)

print("\nFULL model (2024) usable rows:", len(train_full), "/", len(df_train))

# Build X/y for REDUCED model (used for 2025 + future UI mode)
train_reduced = df_train.dropna(subset=REDUCED_FEATURES).copy()
val_reduced   = df_val.dropna(subset=REDUCED_FEATURES).copy()

X_train_red = train_reduced[REDUCED_FEATURES]
y_train_red = train_reduced[TARGET].astype(float)

X_val_red = val_reduced[REDUCED_FEATURES]
y_val_red = val_reduced[TARGET].astype(float)

print("REDUCED train usable rows:", len(train_reduced), "/", len(df_train))
print("REDUCED val usable rows:", len(val_reduced), "/", len(df_val))

# Quick check: how many missing values remain after filtering
print("\nNaNs left in X_train_full:", int(X_train_full.isna().sum().sum()))
print("NaNs left in X_train_red :", int(X_train_red.isna().sum().sum()))
print("NaNs left in X_val_red   :", int(X_val_red.isna().sum().sum()))


Raw rows: 182800
Columns: ['time', 'lon', 'lat', 'SIS', 'aod550', 'tcwv', 'elevation', 'slope', 'aspect', 'ndvi', 'brightness', 'albedo', 'nir', 'tcc']
Time range: 2024-01-01 00:00:00 → 2025-04-01 00:00:00
I0 range: 72.95412325351943 → 483.2262583066537
Kt range: 0.06553530877723426 → 0.7571551085805983

Train days: 366  rows: 146400
Val days  : 90  rows: 36000

FULL model (2024) usable rows: 100883 / 146400
REDUCED train usable rows: 146400 / 146400
REDUCED val usable rows: 36000 / 36000

NaNs left in X_train_full: 0
NaNs left in X_train_red : 0
NaNs left in X_val_red   : 0


## 11. Baseline Model – HistGradientBoosting Regressor (HGBR)
Before training a spatial deep learning model (U-Net), a simple baseline model is trained to verify that:
- The fused dataset is correct and usable
- The selected features contain predictive information for solar radiation
- There are no hidden alignment, scaling or NaN issues

### 11.1 Train / validation split

We follow a temporal split, which reflects the real-world forecasting scenario:
- Training: full year 2024
- Validation: Jan–Mar 2025

This avoids data leakage and simulates predicting unseen future days.

In [17]:
# Reuse df from Section 10 (already contains doy_sin, doy_cos, lat_f, lon_f, I0, Kt)
assert "Kt" in df.columns and "I0" in df.columns, "Run Section 10 first (Kt/I0 not found)."

print("Raw rows:", len(df))
print("Columns:", df.columns.tolist())
print("Time range:", df["time"].min(), "→", df["time"].max())

# Define train / validation periods
train_df = df[df["time"].dt.year == 2024].copy()
val_df   = df[(df["time"] >= "2025-01-01") & (df["time"] <= "2025-03-31")].copy()

print("\nTrain days:", train_df["time"].nunique(), " rows:", len(train_df))
print("Val days  :", val_df["time"].nunique(), " rows:", len(val_df))


Raw rows: 182800
Columns: ['time', 'lon', 'lat', 'SIS', 'aod550', 'tcwv', 'elevation', 'slope', 'aspect', 'ndvi', 'brightness', 'albedo', 'nir', 'tcc', 'doy', 'doy_sin', 'doy_cos', 'lat_f', 'lon_f', 'I0', 'Kt']
Time range: 2024-01-01 00:00:00 → 2025-04-01 00:00:00

Train days: 366  rows: 146400
Val days  : 90  rows: 36000


### 11.2 Feature sets

Because Sentinel-2 features are only available for 2024, we use:
- Full (S2 + CAMS + DEM) → for 2024-only checks
- Atmosphere + Terrain (CAMS + DEM) → for real validation on 2025

In [18]:
FULL_2024_FEATURES = [
    # CAMS
    "tcc", "aod550", "tcwv",
    # DEM
    "elevation", "slope", "aspect",
    # S2
    "ndvi", "brightness", "albedo", "nir",
    # time + location
    "doy_sin", "doy_cos", "lat_f", "lon_f"
]

CAMS_DEM_FEATURES = [
    "aod550", "tcwv",
    "elevation", "slope", "aspect",
    "doy_sin", "doy_cos", "lat_f", "lon_f"
]

TARGET = "Kt"

### 11.3 Prepare training matrices

**Baseline A - Full model check (inside 2024)**

We do a time split within 2024 (train Jan–Oct, validate Nov–Dec).

In [19]:
# Reuse df from Section 10 (already has Kt, I0, doy_sin, lat_f, etc.)
assert "Kt" in df.columns and "I0" in df.columns, "Run Section 10 first."

df["time"] = pd.to_datetime(df["time"])

# 2024 only
df_2024 = df[df["time"].dt.year == 2024].copy()

# Time split within 2024
train_2024 = df_2024[df_2024["time"] <= "2024-10-31"].copy()
val_2024   = df_2024[df_2024["time"] >= "2024-11-01"].copy()

# Drop NaNs 
train_2024_full = train_2024.dropna(subset=FULL_2024_FEATURES + [TARGET])
val_2024_full   = val_2024.dropna(subset=FULL_2024_FEATURES + [TARGET])

X_train_2024_full = train_2024_full[FULL_2024_FEATURES]
y_train_2024_full = train_2024_full[TARGET].astype(float)

X_val_2024_full = val_2024_full[FULL_2024_FEATURES]
y_val_2024_full = val_2024_full[TARGET].astype(float)

print("Baseline A (Full features, 2024 only)")
print("Train rows:", len(X_train_2024_full), " Val rows:", len(X_val_2024_full))


Baseline A (Full features, 2024 only)
Train rows: 84079  Val rows: 16804


**Baseline B - Real future validation (train 2024 → validate 2025)**

This uses only features that exist in 2025.

In [20]:
# Train = full 2024
train_df = df[df["time"].dt.year == 2024].copy()

# Val = Jan–Mar 2025
val_df = df[(df["time"] >= "2025-01-01") & (df["time"] <= "2025-03-31")].copy()

train_cd = train_df.dropna(subset=CAMS_DEM_FEATURES + [TARGET])
val_cd   = val_df.dropna(subset=CAMS_DEM_FEATURES + [TARGET])

X_train_cd = train_cd[CAMS_DEM_FEATURES]
y_train_cd = train_cd[TARGET]

X_val_cd = val_cd[CAMS_DEM_FEATURES]
y_val_cd = val_cd[TARGET]

print("\nBaseline B (CAMS+DEM only, 2024→2025)")
print("Train rows:", len(X_train_cd), " Val rows:", len(X_val_cd))



Baseline B (CAMS+DEM only, 2024→2025)
Train rows: 146400  Val rows: 36000


### 11.4 Train HGBR baseline

A HistGradientBoostingRegressor is used because:
- Handles non-linear relationships well
- Works efficiently on large tabular datasets
- Requires minimal preprocessing

In [21]:
def train_and_eval(name, X_train, y_train, X_val, y_val):
    if len(X_val) == 0:
        print(f"\n{name}: ❌ Validation has 0 rows. Cannot evaluate.")
        return None

    model = HistGradientBoostingRegressor(
        max_depth=8,
        learning_rate=0.05,
        max_iter=200,
        random_state=42
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    mse  = mean_squared_error(y_val, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_val, y_pred)

    print(f"\n{name}")
    print(f"RMSE: {rmse:.4f} (Kt)")
    print(f"R²  : {r2:.3f}")

    return model


This baseline step has one purpose: confirm that the fused dataset is valid and informative before investing time in the U-Net. We trained two HistGradientBoostingRegressor (HGBR) baselines:

**Baseline A — “Full features” check (2024 only)**

- Train: Jan–Oct 2024
- Validate: Nov–Dec 2024
- Features: Sentinel-2 + CAMS + DEM + time + location
  
This is a sanity check baseline because 2024 is the only year where Sentinel-2 monthly features (NDVI, brightness, albedo, NIR) exist in our pipeline.

**Baseline B — “Real future validation” (train 2024 → validate 2025)**

- Train: full year 2024
- Validate: Jan–Mar 2025
- Features: CAMS + DEM + time + location (no Sentinel-2, no tcc after 2024)
  
This is the most realistic baseline, because it tests generalization into a new time period where some feature channels are missing or reduced.

In [22]:

    # Run Baseline A: FULL features (train Jan–Oct 2024, val Nov–Dec 2024)
    hgbr_full_2024 = train_and_eval(
        "Baseline A – FULL (S2+CAMS+DEM) | Train: Jan–Oct 2024 | Val: Nov–Dec 2024",
        X_train_2024_full, y_train_2024_full,
        X_val_2024_full, y_val_2024_full
    )
    
    # Run Baseline B: CAMS+DEM (train 2024, val Jan–Mar 2025)
    hgbr_cams_dem = train_and_eval(
        "Baseline B – CAMS+DEM | Train: 2024 | Val: Jan–Mar 2025",
        X_train_cd, y_train_cd,
        X_val_cd, y_val_cd
    )


Baseline A – FULL (S2+CAMS+DEM) | Train: Jan–Oct 2024 | Val: Nov–Dec 2024
RMSE: 0.1178 (Kt)
R²  : 0.437

Baseline B – CAMS+DEM | Train: 2024 | Val: Jan–Mar 2025
RMSE: 0.1717 (Kt)
R²  : 0.357


In [23]:
Kt_pred = hgbr_cams_dem.predict(X_val_cd)
SIS_pred = Kt_pred * val_cd["I0"].values
SIS_true = val_cd["SIS"].values

rmse_sis = np.sqrt(mean_squared_error(SIS_true, SIS_pred))
print("RMSE (SIS, W/m²):", rmse_sis)

RMSE (SIS, W/m²): 31.572147127392675


**Baseline A — “Full features” check (2024 only)**

The model can learn meaningful radiation patterns when all feature sources are available. This strongly suggests:

- The fusion grid alignment is correct
- The engineered time/solar features are working
- Sentinel-2 + CAMS + terrain contain useful signal for daily GHI

**Baseline B — “Real future validation” (train 2024 → validate 2025)**

Performance drops compared to Baseline A, which is expected because:
- Sentinel-2 surface features are missing in 2025
- tcc becomes NaN during the NRT period
  
Even with reduced inputs, the model still achieves a reasonable RMSE in W/m², meaning:
- CAMS + terrain + seasonal time features already explain a decent amount of radiation variability
- The pipeline supports “predict unseen days” behaviour (important for the final UI)

### 11.6 Hyperparameter search for improvement

To check whether the baseline can be improved without changing the data pipeline, a randomized search was applied.

In [24]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import ParameterSampler
from joblib import Parallel, delayed

def eval_one_config(params, X_train, y_train, X_val, y_val, seed=42):
    model = HistGradientBoostingRegressor(
        random_state=seed,
        early_stopping=True,          # helps avoid overfitting + speeds up
        n_iter_no_change=20,
        validation_fraction=0.1,      # internal split from TRAIN only (safe)
        **params
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_val)

    rmse = float(np.sqrt(mean_squared_error(y_val, pred)))
    r2 = float(r2_score(y_val, pred))
    return {**params, "rmse": rmse, "r2": r2}

def randomized_search_hgbr(name, X_train, y_train, X_val, y_val, n_iter=80, n_jobs=4, seed=42):
    rng = np.random.RandomState(seed)

    # Search space (default ranges for HGBR)
    param_dist = {
        "max_depth": [3, 5, 7, 9, None],
        "learning_rate": list(np.geomspace(0.01, 0.2, 10)),
        "max_iter": [200, 400, 800, 1200],
        "min_samples_leaf": [10, 20, 50, 100, 200],
        "l2_regularization": list(np.geomspace(1e-4, 10.0, 10)),
        "max_bins": [64, 128, 255],
    }

    samples = list(ParameterSampler(param_dist, n_iter=n_iter, random_state=rng))

    print(f"\n{name}: randomized search {len(samples)} configs using {n_jobs} jobs...")

    results = Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(eval_one_config)(params, X_train, y_train, X_val, y_val, seed)
        for params in samples
    )

    df_res = pd.DataFrame(results).sort_values("r2", ascending=False).reset_index(drop=True)

    print("\nTop 10 configs:")
    print(df_res.head(10))

    best_params = df_res.loc[0, [c for c in df_res.columns if c not in ["rmse", "r2"]]].to_dict()
    print("\nBest params:", best_params)
    print(f"Best val R2={df_res.loc[0,'r2']:.4f} | RMSE={df_res.loc[0,'rmse']:.4f} (Kt)")

    return df_res, best_params


In [25]:
# Baseline A (FULL 2024: train Jan–Oct, val Nov–Dec)
res_A, best_A = randomized_search_hgbr(
    "Baseline A (FULL 2024)",
    X_train_2024_full, y_train_2024_full,
    X_val_2024_full, y_val_2024_full,
    n_iter=80, n_jobs=4, seed=42
)

# Baseline B (CAMS+DEM: train 2024, val Jan–Mar 2025)
res_B, best_B = randomized_search_hgbr(
    "Baseline B (CAMS+DEM 2024→2025)",
    X_train_cd, y_train_cd,
    X_val_cd, y_val_cd,
    n_iter=120, n_jobs=4, seed=42
)

# Save results
res_A.to_csv("hgbr_randomsearch_baselineA.csv", index=False)
res_B.to_csv("hgbr_randomsearch_baselineB.csv", index=False)
print("Saved CSVs.")



Baseline A (FULL 2024): randomized search 80 configs using 4 jobs...


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:   15.6s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:   24.4s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:   29.2s
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:   40.2s
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:   54.5s
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:  1.1min
[Parallel(n_jobs=4)]: Done  53 tasks      | elapsed:  1.4min
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:  1.8min
[Parallel(n_jobs=4)]: Done  80 out of  80 | elapsed:  2.2min finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.



Top 10 configs:
   min_samples_leaf  max_iter  max_depth  max_bins  learning_rate  \
0                50       200        9.0        64       0.027144   
1                10       800        NaN       255       0.019459   
2                10       200        5.0       128       0.027144   
3                20      1200        NaN        64       0.013950   
4                50       200        7.0        64       0.027144   
5                20       800        5.0       128       0.013950   
6                10       400        3.0        64       0.010000   
7                10      1200        NaN       128       0.013950   
8                50       800        5.0       128       0.010000   
9                20       400        3.0       255       0.010000   

   l2_regularization      rmse        r2  
0           0.774264  0.108895  0.518776  
1           0.000359  0.109324  0.514971  
2          10.000000  0.109540  0.513053  
3           0.001292  0.109689  0.511730  
4       

[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:   13.1s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:   26.9s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:   34.5s
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:   50.1s
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:  1.2min
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:  1.5min
[Parallel(n_jobs=4)]: Done  53 tasks      | elapsed:  2.0min
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:  2.5min
[Parallel(n_jobs=4)]: Done  77 tasks      | elapsed:  3.0min
[Parallel(n_jobs=4)]: Done  90 tasks      | elapsed:  3.5min
[Parallel(n_jobs=4)]: Done 105 tasks      | elapsed:  4.1min



Top 10 configs:
   min_samples_leaf  max_iter  max_depth  max_bins  learning_rate  \
0               100      1200        3.0       128       0.010000   
1                20       400        3.0       255       0.010000   
2               100       200        7.0       255       0.010000   
3               100       800        5.0       255       0.010000   
4                10       200        NaN       128       0.013950   
5               100      1200        3.0       255       0.027144   
6               200       400        NaN       128       0.010000   
7               100      1200        3.0       255       0.027144   
8                10       400        NaN       128       0.010000   
9                20       200        9.0       128       0.010000   

   l2_regularization      rmse        r2  
0           2.782559  0.163082  0.419223  
1           0.016681  0.163670  0.415031  
2           0.774264  0.166672  0.393376  
3           2.782559  0.166751  0.392800  
4       

[Parallel(n_jobs=4)]: Done 120 out of 120 | elapsed:  4.7min finished


In [26]:
def cast_hgbr_params(p: dict) -> dict:
    p = p.copy()

    # int params
    for k in ["max_bins", "max_iter", "min_samples_leaf"]:
        if k in p and p[k] is not None:
            p[k] = int(p[k])

    # max_depth can be int or None
    if "max_depth" in p:
        if pd.isna(p["max_depth"]):
            p["max_depth"] = None
        elif p["max_depth"] is not None:
            p["max_depth"] = int(p["max_depth"])

    # float params
    for k in ["learning_rate", "l2_regularization"]:
        if k in p and p[k] is not None:
            p[k] = float(p[k])

    return p

best_B_cast = cast_hgbr_params(best_B)
print("Best B params (casted):", best_B_cast)

best_model_B = HistGradientBoostingRegressor(
    random_state=42,
    early_stopping=True,
    n_iter_no_change=20,
    validation_fraction=0.1,
    **best_B_cast
)
best_model_B.fit(X_train_cd, y_train_cd)

Kt_pred = best_model_B.predict(X_val_cd)
SIS_pred = Kt_pred * val_cd["I0"].values
SIS_true = val_cd["SIS"].values

rmse_sis = np.sqrt(mean_squared_error(SIS_true, SIS_pred))
print("Baseline B best -> RMSE (SIS, W/m²):", rmse_sis)


Best B params (casted): {'min_samples_leaf': 100, 'max_iter': 1200, 'max_depth': 3, 'max_bins': 128, 'learning_rate': 0.01, 'l2_regularization': 2.782559402207126}
Baseline B best -> RMSE (SIS, W/m²): 31.181758007580882


**Best Baseline A (Full features, 2024 internal split)**

Best validation performance reached:
- R² ≈ 0.519
- RMSE (Kt) ≈ 0.109

**Best Baseline B (CAMS+DEM, 2024→2025)**

Best validation performance reached:
- R² ≈ 0.419
- RMSE (Kt) ≈ 0.163
- Converted back to SIS: RMSE (SIS) ≈ 31.2 W/m²

### 11.7 Key takeaways before moving to U-Net

1. Data fusion is working
   
- All datasets align on the same 0.05° grid (20×20)
- The daily fused dataset is consistent and trainable

2. Two “input modes” are now clearly defined

- Full-input mode (2024): S2 + CAMS + DEM + time
- Reduced-input mode (2025+): CAMS + DEM + time
  
*=> This matters because the UI will support “future date selection” later.*

3. Baseline confirms feasibility of the final app goal

- The model can predict unseen months using seasonal + atmospheric + terrain signals.

### 11.8 What comes next 
Now that the baseline confirms the pipeline is correct, next step we will start with U-Net model to predict a full daily GHI map (20×20) and preparing UI-ready outputs: city mean GHI number, heatmap, simple overlays/explanations.

## 12. U-Net Radiation Map Modelling (20×20 daily GHI maps)

**Goal**
Train a U-Net style CNN to predict a full daily GHI (SIS) map over Eindhoven on the SARAH 0.05° grid.

- Input: daily feature stack with shape (H=20, W=20, C)
- Output: predicted SIS map (20, 20, 1) in W/m²

**Design choice**
We prepare two model pathways:
- Model A (Historical / Daily-input U-Net): trained on 2024, where we have the richest feature set (CAMS + DEM + Sentinel-2 monthly → daily + seasonality).
- Model B (Future / Seasonal-estimate U-Net): planned later for dates where some inputs are missing (e.g., no Sentinel-2 in 2025).

### 12.1 Define helper functions and feature lists
We define the feature channels for Model A, focusing on variables that are available in 2024:
- CAMS: total cloud cover (tcc), aerosol optical depth (aod550), total column water vapour (tcwv)
- DEM: elevation, slope, aspect
- Sentinel-2: NDVI, brightness, albedo, NIR 
- Seasonality: doy_sin, doy_cos (added as constant 20×20 maps per day)

The target is initially SIS (W/m²) from SARAH-3.

In [27]:
import numpy as np
import pandas as pd

# For Model A (historical), we will train on 2024 only.

UNET_FEATURES_A = [
    # CAMS
    "tcc", "aod550", "tcwv",
    # DEM (static but repeated over time)
    "elevation", "slope", "aspect",
    # Sentinel-2 monthly->daily (available for 2024 only)
    "ndvi", "brightness", "albedo", "nir",
    # Seasonality channels (broadcast later as constant maps)
    "doy_sin", "doy_cos",
]

UNET_TARGET = "SIS"   

def assert_vars_exist(ds, vars_list):
    missing = [v for v in vars_list if v not in ds.data_vars and v not in ds.coords]
    if missing:
        raise ValueError(f"Missing variables in dataset: {missing}")

print("✅ U-Net feature channels (Model A):", len(UNET_FEATURES_A))

✅ U-Net feature channels (Model A): 12


### 12.2 Build training tensors (Model A, 2024)

We convert the fused xarray dataset into CNN-ready tensors:
- X_A: (N_days, 20, 20, C) input feature maps
- y_A: (N_days, 20, 20, 1) target SIS maps

Important handling:
- We use only 2024 (Sentinel-2 surface features exist only there).
- doy_sin / doy_cos are scalars per day so we broadcast them to constant 20×20 maps.
- Any NaNs in feature maps are filled using the day’s spatial mean, else a global 2024 mean for that feature.

This avoids throwing away many days due to partial missingness, while keeping values realistic.

In [28]:
assert "fused" in globals(), "Run fusion steps first: `fused` not found."

fused_A = fused.sel(time=slice("2024-01-01", "2024-12-31")).copy()

# Doy features (1D over time)
times = pd.to_datetime(fused_A["time"].values)
doy = pd.Series(times).dt.dayofyear.values.astype(int)

fused_A = fused_A.assign(
    doy_sin=("time", np.sin(2 * np.pi * doy / 365.25).astype("float32")),
    doy_cos=("time", np.cos(2 * np.pi * doy / 365.25).astype("float32")),
)

H = fused_A.sizes["lat"]
W = fused_A.sizes["lon"]
C = len(UNET_FEATURES_A)

print("Grid:", H, "x", W, "| Channels:", C, "| Days:", fused_A.sizes["time"])

# Global fallback means per channel (computed over 2024)
global_means = {}
for feat in UNET_FEATURES_A:
    if feat in ["doy_sin", "doy_cos"]:
        continue
    vals = fused_A[feat].values.astype("float32")  # (time, lat, lon)
    global_means[feat] = float(np.nanmean(vals))

print("✅ Global feature means computed (NaN-safe).")

# Build tensors
X_list, y_list, kept_times = [], [], []

skipped_y_nan = 0
filled_cells_total = 0

for t in fused_A["time"].values:
    day = fused_A.sel(time=t)

    # Target
    y_day = day[UNET_TARGET].values.astype("float32")  # (H, W)
    if np.isnan(y_day).any():
        skipped_y_nan += 1
        continue

    # Broadcast doy maps
    sin_map = np.full((H, W), float(day["doy_sin"].values), dtype="float32")
    cos_map = np.full((H, W), float(day["doy_cos"].values), dtype="float32")

    channels = []
    for feat in UNET_FEATURES_A:
        if feat == "doy_sin":
            ch = sin_map
        elif feat == "doy_cos":
            ch = cos_map
        else:
            ch = day[feat].values.astype("float32")

            # Fill NaNs in this channel
            if np.isnan(ch).any():
                nan_mask = np.isnan(ch)
                filled_cells_total += int(nan_mask.sum())

                # Day spatial mean (ignoring NaN)
                day_mean = float(np.nanmean(ch)) if np.isfinite(np.nanmean(ch)) else np.nan

                # Fallback to global mean if day mean invalid (e.g., all NaN)
                fill_value = day_mean if np.isfinite(day_mean) else global_means[feat]
                ch = ch.copy()
                ch[nan_mask] = fill_value

        channels.append(ch)

    X_day = np.stack(channels, axis=-1)  # (H, W, C)

    # Final safety check
    if np.isnan(X_day).any():
        continue

    X_list.append(X_day)
    y_list.append(y_day)
    kept_times.append(pd.to_datetime(t))

X_A = np.stack(X_list, axis=0)                 # (N, H, W, C)
y_A = np.stack(y_list, axis=0)[..., None]      # (N, H, W, 1)
kept_times = pd.to_datetime(kept_times)

print("\n Built Model A tensors:")
print("X_A:", X_A.shape, " | y_A:", y_A.shape)
print("Skipped days due to NaN in target SIS:", skipped_y_nan)
print("Total filled NaN cells across all feature maps:", filled_cells_total)
print("NaNs in X_A:", np.isnan(X_A).sum(), "| NaNs in y_A:", np.isnan(y_A).sum())

Grid: 20 x 20 | Channels: 12 | Days: 366
✅ Global feature means computed (NaN-safe).

 Built Model A tensors:
X_A: (366, 20, 20, 12)  | y_A: (366, 20, 20, 1)
Skipped days due to NaN in target SIS: 0
Total filled NaN cells across all feature maps: 181569
NaNs in X_A: 0 | NaNs in y_A: 0


### 12.3 Time-based split (realistic “future month” validation)

We use a time-based split to mimic real deployment:
- Train: Jan–Oct 2024
- Validation: Nov–Dec 2024

This avoids leakage across seasons and tests whether the model generalizes to later unseen months.

In [29]:
assert "X_A" in globals() and "y_A" in globals(), "Run 12.2 first (X_A/y_A not found)."
assert "kept_times" in globals(), "Run 12.2 first (kept_times not found)."

kept_times = pd.to_datetime(kept_times)

# sanity: lengths must match
print("Days in X_A:", X_A.shape[0], "| Days in kept_times:", len(kept_times))
assert X_A.shape[0] == len(kept_times), "Mismatch: X_A and kept_times lengths differ."

train_idx = kept_times <= pd.to_datetime("2024-10-31")
val_idx   = kept_times >= pd.to_datetime("2024-11-01")

X_train_A, y_train_A = X_A[train_idx], y_A[train_idx]
X_val_A, y_val_A     = X_A[val_idx], y_A[val_idx]

print("U-Net Model A split:")
print("Train:", X_train_A.shape, y_train_A.shape)
print("Val  :", X_val_A.shape, y_val_A.shape)
print("Train days:", int(train_idx.sum()), "| Val days:", int(val_idx.sum()))
print("Train range:", kept_times[train_idx].min(), "→", kept_times[train_idx].max())
print("Val range  :", kept_times[val_idx].min(), "→", kept_times[val_idx].max())


Days in X_A: 366 | Days in kept_times: 366
U-Net Model A split:
Train: (305, 20, 20, 12) (305, 20, 20, 1)
Val  : (61, 20, 20, 12) (61, 20, 20, 1)
Train days: 305 | Val days: 61
Train range: 2024-01-01 00:00:00 → 2024-10-31 00:00:00
Val range  : 2024-11-01 00:00:00 → 2024-12-31 00:00:00


### 12.4 Normalize inputs (train-only stats) + build TensorFlow datasets
In this step we:
- Compute per-channel mean & std using training data only (to avoid leakage).
- Normalize X_train_A and X_val_A using those train stats.
- Create efficient tf.data.Dataset pipelines for U-Net training.

In [30]:
import numpy as np
import tensorflow as tf

assert "X_train_A" in globals() and "X_val_A" in globals(), "Run 12.3 first."
assert "y_train_A" in globals() and "y_val_A" in globals(), "Run 12.3 first."

# Compute train-only normalization stats (per channel)
# X_train_A: (N, H, W, C)
eps = 1e-6

train_mean = X_train_A.mean(axis=(0, 1, 2), keepdims=True).astype("float32")  # (1,1,1,C)
train_std  = X_train_A.std(axis=(0, 1, 2), keepdims=True).astype("float32")   # (1,1,1,C)
train_std = np.maximum(train_std, eps)

print("train_mean shape:", train_mean.shape, "| train_std shape:", train_std.shape)
print("train_std min/max:", float(train_std.min()), "→", float(train_std.max()))

# Apply normalization
X_train_norm = ((X_train_A - train_mean) / train_std).astype("float32")
X_val_norm   = ((X_val_A   - train_mean) / train_std).astype("float32")

# Targets as float32
y_train_f = y_train_A.astype("float32")
y_val_f   = y_val_A.astype("float32")

print("\n✅ Normalized tensors:")
print("X_train_norm:", X_train_norm.shape, " y_train:", y_train_f.shape)
print("X_val_norm  :", X_val_norm.shape,   " y_val  :", y_val_f.shape)

# Check no NaNs
print("NaNs in X_train_norm:", int(np.isnan(X_train_norm).sum()))
print("NaNs in X_val_norm  :", int(np.isnan(X_val_norm).sum()))
print("NaNs in y_train_f   :", int(np.isnan(y_train_f).sum()))
print("NaNs in y_val_f     :", int(np.isnan(y_val_f).sum()))

# Build tf.data datasets
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

ds_train = tf.data.Dataset.from_tensor_slices((X_train_norm, y_train_f))
ds_train = ds_train.shuffle(buffer_size=len(X_train_norm), reshuffle_each_iteration=True)
ds_train = ds_train.batch(BATCH_SIZE).prefetch(AUTOTUNE)

ds_val = tf.data.Dataset.from_tensor_slices((X_val_norm, y_val_f))
ds_val = ds_val.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# Quick shape check from dataset
for xb, yb in ds_train.take(1):
    print("\nBatch shapes:")
    print("X:", xb.shape, "y:", yb.shape)

print("\n✅ tf.data datasets ready.")

train_mean shape: (1, 1, 1, 12) | train_std shape: (1, 1, 1, 12)
train_std min/max: 0.018138384446501732 → 104.30235290527344

✅ Normalized tensors:
X_train_norm: (305, 20, 20, 12)  y_train: (305, 20, 20, 1)
X_val_norm  : (61, 20, 20, 12)  y_val  : (61, 20, 20, 1)
NaNs in X_train_norm: 0
NaNs in X_val_norm  : 0
NaNs in y_train_f   : 0
NaNs in y_val_f     : 0

Batch shapes:
X: (16, 20, 20, 12) y: (16, 20, 20, 1)

✅ tf.data datasets ready.


### 12.5 Build a lightweight U-Net (Small)

We start with a small U-Net (2 downsampling levels) because:
- The grid is only 20×20
- The dataset is relatively small (366 samples)
- We want a stable baseline before increasing capacity.

The model outputs a single regression channel (20, 20, 1).

In [31]:
import tensorflow as tf
from tensorflow.keras import layers, Model

assert "X_train_norm" in globals(), "Run 12.4 first."

H, W, C = X_train_norm.shape[1], X_train_norm.shape[2], X_train_norm.shape[3]
print("Input shape:", (H, W, C))

def conv_block(x, filters, name):
    x = layers.Conv2D(filters, 3, padding="same", activation="relu", name=f"{name}_conv1")(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="relu", name=f"{name}_conv2")(x)
    return x

def build_unet_small(input_shape=(20, 20, 12), base=32):
    inputs = layers.Input(shape=input_shape, name="input")

    # Encoder
    c1 = conv_block(inputs, base, "enc1")
    p1 = layers.MaxPool2D(pool_size=(2, 2), name="pool1")(c1)   # 20->10

    c2 = conv_block(p1, base * 2, "enc2")
    p2 = layers.MaxPool2D(pool_size=(2, 2), name="pool2")(c2)   # 10->5

    # Bottleneck
    b = conv_block(p2, base * 4, "bottleneck")                  # 5x5

    # Decoder
    u2 = layers.UpSampling2D(size=(2, 2), name="up2")(b)        # 5->10
    u2 = layers.Concatenate(name="skip2")([u2, c2])
    c3 = conv_block(u2, base * 2, "dec2")

    u1 = layers.UpSampling2D(size=(2, 2), name="up1")(c3)       # 10->20
    u1 = layers.Concatenate(name="skip1")([u1, c1])
    c4 = conv_block(u1, base, "dec1")

    # Output regression map
    outputs = layers.Conv2D(1, 1, padding="same", activation="linear", name="sis_pred")(c4)

    return Model(inputs, outputs, name="UNetSmall")

model_unet = build_unet_small(input_shape=(H, W, C), base=32)
model_unet.summary()

# Loss + metrics
model_unet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.MeanAbsoluteError(),
    metrics=[
        tf.keras.metrics.RootMeanSquaredError(name="rmse"),
        tf.keras.metrics.MeanAbsoluteError(name="mae"),
    ],
)

print("\n Model compiled.")


Input shape: (20, 20, 12)


Model: "UNetSmall"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)            │ (None, 20, 20, 12)        │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ enc1_conv1 (Conv2D)           │ (None, 20, 20, 32)        │           3,488 │ input[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ enc1_conv2 (Conv2D)           │ (None, 20, 20, 32)        │           9,248 │ enc1_conv1[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1 (MaxPooling2D)          │ (None, 10, 10, 32)        │               0 │ enc1_conv2[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ enc2_conv1 (Conv2D)           │ (None, 10, 10, 64)        │          18,496 │ pool1[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ enc2_conv2 (Conv2D)           │ (None, 10, 10, 64)        │          36,928 │ enc2_conv1[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool2 (MaxPooling2D)          │ (None, 5, 5, 64)          │               0 │ enc2_conv2[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bottleneck_conv1 (Conv2D)     │ (None, 5, 5, 128)         │          73,856 │ pool2[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bottleneck_conv2 (Conv2D)     │ (None, 5, 5, 128)         │         147,584 │ bottleneck_conv1[0][0]     │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ up2 (UpSampling2D)            │ (None, 10, 10, 128)       │               0 │ bottleneck_conv2[0][0]     │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ skip2 (Concatenate)           │ (None, 10, 10, 192)       │               0 │ up2[0][0],                 │
│                               │                           │                 │ enc2_conv2[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dec2_conv1 (Conv2D)           │ (None, 10, 10, 64)        │         110,656 │ skip2[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dec2_conv2 (Conv2D)           │ (None, 10, 10, 64)        │          36,928 │ dec2_conv1[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ up1 (UpSampling2D)            │ (None, 20, 20, 64)        │               0 │ dec2_conv2[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ skip1 (Concatenate)           │ (None, 20, 20, 96)        │               0 │ up1[0][0],                 │
│                               │                           │                 │ enc1_conv2[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dec1_conv1 (Conv2D)           │ (None, 20, 20, 32)        │          27,680 │ skip1[0][0]                │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 474,145 (1.81 MB)

 Trainable params: 474,145 (1.81 MB)

 Non-trainable params: 0 (0.00 B)


 Model compiled.


### 12.6 Train Model A (direct SIS target)

We train with:
- EarlyStopping to prevent overfitting
- ReduceLROnPlateau to improve convergence when validation stalls
- ModelCheckpoint to save the best model

This run is mainly a baseline sanity check: does the CNN learn anything meaningful from the fused feature maps?

In [32]:
from pathlib import Path
import tensorflow as tf

assert "model_unet" in globals(), "Run 12.5 first (model_unet missing)."
assert "X_train_norm" in globals() and "X_val_norm" in globals(), "Run 12.4 first (normalized X missing)."
assert "y_train_A" in globals() and "y_val_A" in globals(), "y_train_A / y_val_A missing."

# Ensure float32
X_train = X_train_norm.astype("float32")
X_val   = X_val_norm.astype("float32")
y_train = y_train_A.astype("float32")
y_val   = y_val_A.astype("float32")

print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)

# Where to save best model
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)
BEST_PATH = MODEL_DIR / "unet_modelA_best.keras"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_PATH),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=12,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
]

history = model_unet.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=120,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Training finished.")
print("Best model path:", BEST_PATH)

Train: (305, 20, 20, 12) (305, 20, 20, 1)
Val  : (61, 20, 20, 12) (61, 20, 20, 1)
Epoch 1/120
19/20 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 108.9286 - mae: 108.9287 - rmse: 135.7525
Epoch 1: val_loss improved from None to 53.32504, saving model to models\unet_modelA_best.keras
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - loss: 83.5269 - mae: 83.5269 - rmse: 110.1777 - val_loss: 53.3250 - val_mae: 53.3250 - val_rmse: 69.3056 - learning_rate: 0.0010
Epoch 2/120
19/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 54.0897 - mae: 54.0897 - rmse: 72.6893
Epoch 2: val_loss improved from 53.32504 to 17.39760, saving model to models\unet_modelA_best.keras
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 51.3297 - mae: 51.3297 - rmse: 68.6777 - val_loss: 17.3976 - val_mae: 17.3976 - val_rmse: 21.7758 - learning_rate: 0.0010
Epoch 3/120
19/20 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 62.7879 - mae: 62.7879 - rmse: 80.3268   
Epoch 3: val_loss did not improve from 17.39760
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 

### 12.7 Evaluate Model A (direct SIS prediction)

We evaluate in two ways:
- Pixel-wise metrics across all validation pixels (strict, harder)
- City-mean diagnostics (what UI will show later)

In [33]:
# 12.7 Evaluation + visualization (Model A)

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

assert "X_val_norm" in globals() and "y_val_A" in globals(), "Need X_val_norm and y_val_A"
assert "kept_times" in globals(), "Need kept_times from 12.2"
assert "val_idx" in globals(), "Need val_idx from 12.3"
assert "BEST_PATH" in globals(), "Need BEST_PATH from 12.6"

# Load best model
best_model = tf.keras.models.load_model(str(BEST_PATH), compile=False)
print("✅ Loaded best model:", BEST_PATH)

# Predict on validation
y_pred = best_model.predict(X_val_norm, verbose=0)  # (N,20,20,1)

y_true = y_val_A.astype("float32")

# Flatten for metrics
yt = y_true.reshape((y_true.shape[0], -1))
yp = y_pred.reshape((y_pred.shape[0], -1))

mae = mean_absolute_error(yt, yp)
rmse = np.sqrt(mean_squared_error(yt, yp))

print(f"\nValidation metrics (pixel-wise, SIS W/m²):")
print(f"MAE : {mae:.3f} W/m²")
print(f"RMSE: {rmse:.3f} W/m²")

# Build the time list for validation samples (same order as X_val_norm)
val_times = kept_times[val_idx]
assert len(val_times) == len(X_val_norm), "Mismatch: val_times vs X_val_norm length"

i = 0  # change to any index (0..len(val_times)-1)
date_i = val_times[i]

true_map = y_true[i, ..., 0]
pred_map = y_pred[i, ..., 0]
err_map  = pred_map - true_map

true_mean = float(true_map.mean())
pred_mean = float(pred_map.mean())

print(f"\nExample day: {date_i}")
print(f"City mean SIS (true): {true_mean:.2f} W/m²")
print(f"City mean SIS (pred): {pred_mean:.2f} W/m²")
print(f"Mean bias (pred-true): {pred_mean-true_mean:.2f} W/m²")

✅ Loaded best model: models\unet_modelA_best.keras

Validation metrics (pixel-wise, SIS W/m²):
MAE : 17.398 W/m²
RMSE: 21.776 W/m²

Example day: 2024-11-01 00:00:00
City mean SIS (true): 18.81 W/m²
City mean SIS (pred): 15.41 W/m²
Mean bias (pred-true): -3.40 W/m²


In [34]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

assert "y_pred" in globals(), "Run 12.7 first to produce y_pred"
assert "y_val_A" in globals(), "Need y_val_A"
assert "val_times" in globals(), "Need val_times from 12.7"

y_true = y_val_A.astype("float32")          # (N,20,20,1)
y_hat  = y_pred.astype("float32")           # (N,20,20,1)

# Pixel-wise R² across ALL pixels of ALL validation days 
yt_pix = y_true.reshape((y_true.shape[0], -1))
yh_pix = y_hat.reshape((y_hat.shape[0], -1))

r2_pix = r2_score(yt_pix.ravel(), yh_pix.ravel())

# City mean series 
true_mean = yt_pix.mean(axis=1)   # (N,)
pred_mean = yh_pix.mean(axis=1)   # (N,)

r2_mean   = r2_score(true_mean, pred_mean)
mae_mean  = mean_absolute_error(true_mean, pred_mean)
rmse_mean = np.sqrt(mean_squared_error(true_mean, pred_mean))
bias_mean = float((pred_mean - true_mean).mean())

print("Validation metrics:")
print(f"Pixel-wise R² (all pixels): {r2_pix:.4f}")
print(f"City-mean R² (daily mean) : {r2_mean:.4f}")
print(f"City-mean MAE             : {mae_mean:.3f} W/m²")
print(f"City-mean RMSE            : {rmse_mean:.3f} W/m²")
print(f"City-mean bias (pred-true): {bias_mean:.3f} W/m²")

# Monthly bias check (Nov–Dec 2024)
df_eval = pd.DataFrame({
    "time": pd.to_datetime(val_times),
    "true_mean": true_mean,
    "pred_mean": pred_mean,
})
df_eval["err"] = df_eval["pred_mean"] - df_eval["true_mean"]
df_eval["month"] = df_eval["time"].dt.to_period("M").astype(str)

month_stats = df_eval.groupby("month").agg(
    days=("err", "size"),
    mean_true=("true_mean", "mean"),
    mean_pred=("pred_mean", "mean"),
    bias=("err", "mean"),
    mae=("err", lambda x: float(np.mean(np.abs(x)))),
).reset_index()

print("\nMonthly city-mean stats (validation):")
print(month_stats)

# Show worst 5 days by absolute error
worst = df_eval.reindex(df_eval["err"].abs().sort_values(ascending=False).index).head(5)
print("\nWorst 5 validation days (city-mean abs error):")
print(worst[["time", "true_mean", "pred_mean", "err"]])

Validation metrics:
Pixel-wise R² (all pixels): -0.5105
City-mean R² (daily mean) : -0.4575
City-mean MAE             : 16.980 W/m²
City-mean RMSE            : 20.530 W/m²
City-mean bias (pred-true): -3.302 W/m²

Monthly city-mean stats (validation):
     month  days  mean_true  mean_pred       bias        mae
0  2024-11    30  35.203251  15.130541 -20.072710  20.072710
1  2024-12    31  18.482178  31.409622  12.927446  13.986449

Worst 5 validation days (city-mean abs error):
         time  true_mean  pred_mean        err
3  2024-11-04  83.192497  25.509415 -57.683083
4  2024-11-05  83.690002  26.695757 -56.994247
2  2024-11-03  65.387497  22.577431 -42.810066
10 2024-11-11  57.322498  15.467495 -41.855003
17 2024-11-18  46.207500  10.564445 -35.643055


**Observation:** Direct SIS prediction gives relatively weak generalization (especially city-mean).
=> Reason: SIS magnitude is dominated by extraterrestrial radiation (I₀) (seasonal geometry). So the model is forced to learn both: “how strong the sun is” (geometry) and “how much reaches the surface” (atmosphere)

This motivates switching to a physically-informed target: clearness index Kt, where: Kt = SIS / I₀

Kt removes most seasonal scaling and focuses learning on atmospheric modulation.

### 12.8 Build Kt targets (physically-informed)

We compute extraterrestrial radiation I₀ using latitude + day-of-year then convert SIS → Kt.
We train the CNN on Kt and later reconstruct SIS as: SIS(pred) = Kt(pred) x I₀

This is consistent with the HGBR baseline approach and usually improves generalization.

In [35]:
import numpy as np
import pandas as pd

# --- helper: extraterrestrial radiation (daily mean W/m²) ---
def extraterrestrial_radiation_Wm2(lat_deg, doy):
    lat = np.deg2rad(lat_deg.astype(float))
    doy = doy.astype(float)

    dr = 1 + 0.033 * np.cos(2 * np.pi * doy / 365.0)
    delta = 0.409 * np.sin(2 * np.pi * doy / 365.0 - 1.39)
    ws = np.arccos(np.clip(-np.tan(lat) * np.tan(delta), -1, 1))

    Gsc = 1367.0
    Ra = (24 * 3600 / np.pi) * Gsc * dr * (
        ws * np.sin(lat) * np.sin(delta) +
        np.cos(lat) * np.cos(delta) * np.sin(ws)
    )
    return Ra / (24 * 3600)

assert "kept_times" in globals(), "kept_times not found. Run 12.2 first."
assert "fused_A" in globals(), "fused_A not found. Run 12.2 first."
assert len(kept_times) == y_A.shape[0], f"Mismatch: len(kept_times)={len(kept_times)} vs y_A days={y_A.shape[0]}"

# Grid lat map (H,W)
lat_vec = fused_A["lat"].values.astype("float32")
H = fused_A.sizes["lat"]
W = fused_A.sizes["lon"]
lat_map = np.repeat(lat_vec[:, None], W, axis=1).astype("float32")  # (H,W)

# Compute doy per kept day
doy = pd.to_datetime(kept_times).dayofyear.values.astype(int)

# I0 per day (scalar per day, same for all lon at this small extent; varies slightly with lat)
# We'll compute I0_map per day using lat_map and doy:
I0_maps = []
for d in doy:
    I0_day = extraterrestrial_radiation_Wm2(lat_map, np.full_like(lat_map, d))
    I0_maps.append(I0_day.astype("float32"))
I0_maps = np.stack(I0_maps, axis=0)  # (N,H,W)

# Build Kt target from SIS target (y_A is SIS)
eps = 1e-6
SIS = y_A[..., 0].astype("float32")  # (N,H,W)
Kt = SIS / (I0_maps + eps)

# Clip to physical-ish bounds (same idea as your baseline)
Kt = np.clip(Kt, 0.0, 1.2).astype("float32")

y_Kt = Kt[..., None]  # (N,H,W,1)

print("✅ Built y_Kt:", y_Kt.shape, "| range:", float(y_Kt.min()), "→", float(y_Kt.max()))
print("NaNs in y_Kt:", int(np.isnan(y_Kt).sum()))

✅ Built y_Kt: (366, 20, 20, 1) | range: 0.0759730264544487 → 0.7445186972618103
NaNs in y_Kt: 0


### 12.9 Split train / validation set

In [36]:
assert "train_idx" in globals() and "val_idx" in globals(), "train_idx/val_idx not found. Use your fixed 12.3 split."

y_train_Kt = y_Kt[train_idx].astype("float32")
y_val_Kt   = y_Kt[val_idx].astype("float32")

print("Train Kt:", y_train_Kt.shape, "| Val Kt:", y_val_Kt.shape)
print("NaNs train/val:", int(np.isnan(y_train_Kt).sum()), int(np.isnan(y_val_Kt).sum()))

Train Kt: (305, 20, 20, 1) | Val Kt: (61, 20, 20, 1)
NaNs train/val: 0 0


### 12.10 Train small U-Net on Kt (MAE loss)

We reuse the same U-Net architecture but change the target to Kt.
After predicting Kt on validation, we convert back to SIS for evaluation.

Expected improvement:
- Better seasonal generalization
- Stronger city-mean R²
- Reduced bias drift

In [37]:
import tensorflow as tf
from tensorflow import keras
import os

input_shape = X_train_norm.shape[1:]  # (20, 20, 12)

# Build model (same architecture as before)
model_Kt = build_unet_small(input_shape)

model_Kt.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="mae",
    metrics=[
        keras.metrics.MeanAbsoluteError(name="mae"),
        keras.metrics.RootMeanSquaredError(name="rmse")
    ]
)

os.makedirs("models", exist_ok=True)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        "models/unet_modelA_Kt_best.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-5,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=12,
        restore_best_weights=True,
        verbose=1
    )
]

history_Kt = model_Kt.fit(
    X_train_norm, y_train_Kt,
    validation_data=(X_val_norm, y_val_Kt),
    epochs=120,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)

print("✅ Training finished (Kt). Best model: models/unet_modelA_Kt_best.keras")

Epoch 1/120
19/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.2247 - mae: 0.2247 - rmse: 0.2833
Epoch 1: val_loss improved from None to 0.17678, saving model to models/unet_modelA_Kt_best.keras
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 43ms/step - loss: 0.1662 - mae: 0.1662 - rmse: 0.2181 - val_loss: 0.1768 - val_mae: 0.1768 - val_rmse: 0.2082 - learning_rate: 0.0010
Epoch 2/120
19/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.1330 - mae: 0.1330 - rmse: 0.1669
Epoch 2: val_loss improved from 0.17678 to 0.11536, saving model to models/unet_modelA_Kt_best.keras
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.1255 - mae: 0.1255 - rmse: 0.1579 - val_loss: 0.1154 - val_mae: 0.1154 - val_rmse: 0.1527 - learning_rate: 0.0010
Epoch 3/120
19/20 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.1222 - mae: 0.1222 - rmse: 0.1563
Epoch 3: val_loss did not improve from 0.11536
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.1147 - mae: 0.1147 - rmse: 0.1459 - val_loss: 0.1214 - val_mae: 0.1214 - val_rmse: 0.148

### 12.11 Evaluate Kt model (convert back to SIS)

Even though the model predicts Kt, we convert predictions back to SIS (W/m²) because that is what we compare to SARAH and what the UI will show.

We evaluate:
- Pixel-wise MAE / RMSE / R² (all pixels pooled across all validation days)
- City-mean per day MAE / RMSE / R² + bias (mean(pred−true))
- Monthly bias + MAE sanity check (Nov vs Dec)
- Worst days by absolute city-mean error (to see “failure cases”)

**Why we expect some hard days**

Winter months include many low-sun or cloudy days. This creates:
- Many low SIS values (harder regression near 0)
- Occasional spikes (clear-sky days) that can dominate errors

So we focus on bias and worst-day diagnostics, not only average MAE.

In [38]:
import numpy as np
import pandas as pd
from tensorflow import keras
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

best_Kt = keras.models.load_model("models/unet_modelA_Kt_best.keras")
print("✅ Loaded:", "models/unet_modelA_Kt_best.keras")

# Predict Kt on val
Kt_pred = best_Kt.predict(X_val_norm, verbose=0).astype("float32")   # (Nval,H,W,1)
Kt_true = y_val_Kt.astype("float32")

# Get matching I0_maps for val subset
I0_val = I0_maps[val_idx].astype("float32")  # (Nval,H,W)

# Convert to SIS
SIS_pred = (Kt_pred[..., 0] * I0_val).astype("float32")
SIS_true = (y_A[val_idx][..., 0]).astype("float32")  # original SIS target

# --- pixel-wise metrics (all pixels pooled) ---
SIS_true_flat = SIS_true.reshape(-1)
SIS_pred_flat = SIS_pred.reshape(-1)

pixel_mae  = mean_absolute_error(SIS_true_flat, SIS_pred_flat)
pixel_rmse = np.sqrt(mean_squared_error(SIS_true_flat, SIS_pred_flat))
pixel_r2   = r2_score(SIS_true_flat, SIS_pred_flat)

# --- city-mean per day metrics ---
true_mean = SIS_true.mean(axis=(1,2))
pred_mean = SIS_pred.mean(axis=(1,2))

city_mae  = mean_absolute_error(true_mean, pred_mean)
city_rmse = np.sqrt(mean_squared_error(true_mean, pred_mean))
city_r2   = r2_score(true_mean, pred_mean)
city_bias = float((pred_mean - true_mean).mean())

print("\nValidation metrics (SIS W/m²):")
print(f"Pixel-wise  MAE : {pixel_mae:.3f}")
print(f"Pixel-wise  RMSE: {pixel_rmse:.3f}")
print(f"Pixel-wise  R²  : {pixel_r2:.4f}")
print(f"City-mean   MAE : {city_mae:.3f}")
print(f"City-mean   RMSE: {city_rmse:.3f}")
print(f"City-mean   R²  : {city_r2:.4f}")
print(f"City-mean   bias: {city_bias:.3f}")

# Monthly bias/MAE table (val months)
val_times = pd.to_datetime(kept_times[val_idx])
dfm = pd.DataFrame({
    "time": val_times,
    "month": val_times.strftime("%Y-%m"),
    "true_mean": true_mean,
    "pred_mean": pred_mean
})
dfm["bias"] = dfm["pred_mean"] - dfm["true_mean"]
dfm["abs_err"] = np.abs(dfm["bias"])

monthly = dfm.groupby("month").agg(
    days=("time", "count"),
    mean_true=("true_mean", "mean"),
    mean_pred=("pred_mean", "mean"),
    bias=("bias", "mean"),
    mae=("abs_err", "mean")
).reset_index()

print("\nMonthly city-mean stats (validation):")
print(monthly)

# Worst 5 days by abs error
worst5 = dfm.sort_values("abs_err", ascending=False).head(5)[["time","true_mean","pred_mean","bias","abs_err"]]
print("\nWorst 5 validation days (city-mean abs error):")
print(worst5)


✅ Loaded: models/unet_modelA_Kt_best.keras

Validation metrics (SIS W/m²):
Pixel-wise  MAE : 9.291
Pixel-wise  RMSE: 12.242
Pixel-wise  R²  : 0.5226
City-mean   MAE : 7.997
City-mean   RMSE: 10.511
City-mean   R²  : 0.6180
City-mean   bias: -0.235

Monthly city-mean stats (validation):
     month  days  mean_true  mean_pred      bias        mae
0  2024-11    30  35.203251  34.664478 -0.538772  11.002537
1  2024-12    31  18.482178  18.540960  0.058784   5.088942

Worst 5 validation days (city-mean abs error):
         time  true_mean  pred_mean       bias    abs_err
4  2024-11-05  83.690002  55.465565 -28.224438  28.224438
0  2024-11-01  18.807501  44.494434  25.686934  25.686934
3  2024-11-04  83.192497  58.737213 -24.455284  24.455284
7  2024-11-08  18.400000  38.413040  20.013041  20.013041
30 2024-12-01  46.317501  26.307701 -20.009800  20.009800


**Observations** 

Compared to direct SIS training, predicting Kt greatly improves generalization:

- Pixel-wise: MAE 9.291 W/m², R² 0.5226 (vs negative R² for direct SIS)
- City-mean:  MAE 7.997 W/m², R² 0.6180
- Overall city-mean bias is close to zero: −0.235 W/m²

Monthly behaviour is fairly balanced:
- November bias = −0.539 W/m² (slight underprediction)
- December bias = +0.059 W/m² (almost unbiased)

However, worst-day errors remain large (±20–28 W/m²). These “hard days” dominate perception in a map UI because the full map looks clearly wrong even if the average MAE is acceptable.

**Decision: switch from MAE → Huber (robust regression)**

MAE treats every error linearly so a few extreme days can dominate training dynamics and encourage regression-to-the-mean behaviour.
Huber loss is quadratic near zero (better gradient shaping) and becomes linear for larger errors, which often stabilizes training and reduces the influence of extreme outliers.


### 12.12 Train Kt U-Net with Huber loss (more robust to outliers)

In [39]:
import tensorflow as tf
from tensorflow import keras
import os

input_shape = X_train_norm.shape[1:]
model_Kt_huber = build_unet_small(input_shape)

model_Kt_huber.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.Huber(delta=0.05),   # delta in Kt units; 0.03–0.08 is a good range
    metrics=[
        keras.metrics.MeanAbsoluteError(name="mae"),
        keras.metrics.RootMeanSquaredError(name="rmse")
    ]
)

os.makedirs("models", exist_ok=True)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        "models/unet_modelA_Kt_huber_best.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=6,
        min_lr=1e-5,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True,
        verbose=1
    )
]

history_huber = model_Kt_huber.fit(
    X_train_norm, y_train_Kt,
    validation_data=(X_val_norm, y_val_Kt),
    epochs=200,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)

print("✅ Done. Best model: models/unet_modelA_Kt_huber_best.keras")

Epoch 1/200
19/20 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0122 - mae: 0.2673 - rmse: 0.3373
Epoch 1: val_loss improved from None to 0.00529, saving model to models/unet_modelA_Kt_huber_best.keras
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 44ms/step - loss: 0.0088 - mae: 0.1987 - rmse: 0.2622 - val_loss: 0.0053 - val_mae: 0.1284 - val_rmse: 0.1786 - learning_rate: 0.0010
Epoch 2/200
19/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0054 - mae: 0.1314 - rmse: 0.1641
Epoch 2: val_loss improved from 0.00529 to 0.00485, saving model to models/unet_modelA_Kt_huber_best.keras
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0052 - mae: 0.1266 - rmse: 0.1583 - val_loss: 0.0048 - val_mae: 0.1198 - val_rmse: 0.1582 - learning_rate: 0.0010
Epoch 3/200
19/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0044 - mae: 0.1100 - rmse: 0.1392
Epoch 3: val_loss did not improve from 0.00485
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.0045 - mae: 0.1121 - rmse: 0.1395 - val_loss: 0.0050 - val_mae: 0.1222 - val

### 12.13 Evaluate Huber-Kt model in SIS units + observations

In [40]:
import numpy as np
import pandas as pd
from tensorflow import keras
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

best_Kt = keras.models.load_model("models/unet_modelA_Kt_huber_best.keras")
print("✅ Loaded:", "models/unet_modelA_Kt_huber_best.keras")

# Predict Kt on val
Kt_pred = best_Kt.predict(X_val_norm, verbose=0).astype("float32")   # (Nval,H,W,1)
Kt_true = y_val_Kt.astype("float32")

# Get matching I0_maps for val subset
I0_val = I0_maps[val_idx].astype("float32")  # (Nval,H,W)

# Convert to SIS
SIS_pred = (Kt_pred[..., 0] * I0_val).astype("float32")
SIS_true = (y_A[val_idx][..., 0]).astype("float32")  # original SIS target

# --- pixel-wise metrics (all pixels pooled) ---
SIS_true_flat = SIS_true.reshape(-1)
SIS_pred_flat = SIS_pred.reshape(-1)

pixel_mae  = mean_absolute_error(SIS_true_flat, SIS_pred_flat)
pixel_rmse = np.sqrt(mean_squared_error(SIS_true_flat, SIS_pred_flat))
pixel_r2   = r2_score(SIS_true_flat, SIS_pred_flat)

# --- city-mean per day metrics ---
true_mean = SIS_true.mean(axis=(1,2))
pred_mean = SIS_pred.mean(axis=(1,2))

city_mae  = mean_absolute_error(true_mean, pred_mean)
city_rmse = np.sqrt(mean_squared_error(true_mean, pred_mean))
city_r2   = r2_score(true_mean, pred_mean)
city_bias = float((pred_mean - true_mean).mean())

print("\nValidation metrics (SIS W/m²):")
print(f"Pixel-wise  MAE : {pixel_mae:.3f}")
print(f"Pixel-wise  RMSE: {pixel_rmse:.3f}")
print(f"Pixel-wise  R²  : {pixel_r2:.4f}")
print(f"City-mean   MAE : {city_mae:.3f}")
print(f"City-mean   RMSE: {city_rmse:.3f}")
print(f"City-mean   R²  : {city_r2:.4f}")
print(f"City-mean   bias: {city_bias:.3f}")

# Monthly bias/MAE table (val months)
val_times = pd.to_datetime(kept_times[val_idx])
dfm = pd.DataFrame({
    "time": val_times,
    "month": val_times.strftime("%Y-%m"),
    "true_mean": true_mean,
    "pred_mean": pred_mean
})
dfm["bias"] = dfm["pred_mean"] - dfm["true_mean"]
dfm["abs_err"] = np.abs(dfm["bias"])

monthly = dfm.groupby("month").agg(
    days=("time", "count"),
    mean_true=("true_mean", "mean"),
    mean_pred=("pred_mean", "mean"),
    bias=("bias", "mean"),
    mae=("abs_err", "mean")
).reset_index()

print("\nMonthly city-mean stats (validation):")
print(monthly)

# Worst 5 days by abs error
worst5 = dfm.sort_values("abs_err", ascending=False).head(5)[["time","true_mean","pred_mean","bias","abs_err"]]
print("\nWorst 5 validation days (city-mean abs error):")
print(worst5)


✅ Loaded: models/unet_modelA_Kt_huber_best.keras

Validation metrics (SIS W/m²):
Pixel-wise  MAE : 8.109
Pixel-wise  RMSE: 11.346
Pixel-wise  R²  : 0.5899
City-mean   MAE : 6.407
City-mean   RMSE: 9.242
City-mean   R²  : 0.7047
City-mean   bias: -1.619

Monthly city-mean stats (validation):
     month  days  mean_true  mean_pred      bias       mae
0  2024-11    30  35.203251  33.235073 -1.968175  9.052694
1  2024-12    31  18.482178  17.200264 -1.281914  3.846175

Worst 5 validation days (city-mean abs error):
         time  true_mean  pred_mean       bias    abs_err
4  2024-11-05  83.690002  59.102089 -24.587914  24.587914
0  2024-11-01  18.807501  42.989918  24.182417  24.182417
17 2024-11-18  46.207500  23.714706 -22.492794  22.492794
49 2024-12-20  37.087502  15.778067 -21.309435  21.309435
23 2024-11-24  38.597500  20.624704 -17.972795  17.972795


**Observation**

Overall, Huber improves performance compared to the MAE-Kt run, especially on average metrics:

- Pixel-wise: MAE 8.109 W/m², RMSE 11.346, R² 0.5899
- City-mean:  MAE 6.407 W/m², RMSE 9.242, R² 0.7047

Calibration note:
- Overall city-mean bias is negative: −1.619 W/m² → the model slightly underestimates SIS on average.

Monthly behaviour:
- November bias = −1.968 W/m² (underprediction)
- December bias = −1.282 W/m² (still underpredicting but slightly less negative)

Worst-day diagnostics show large errors remain:
- 2024-11-05: true_mean 83.69 vs pred_mean 59.10 → underprediction −24.59 W/m²
- 2024-11-01: true_mean 18.81 vs pred_mean 42.99 → overprediction +24.18 W/m²
These extremes indicate that while Huber stabilizes training, the small U-Net still struggles on unusual atmospheric days and high/low extremes.

**Decision:**
Keep the robust Huber objective but increase model capacity (UNetMedium) to better represent complex spatial patterns and reduce extreme-day failures.


### 12.14 Increase model capacity (UNetMedium) + evaluate (still Huber)

In [41]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def build_unet_medium(input_shape, base=48):
    inputs = keras.Input(shape=input_shape)

    # Encoder 1
    x1 = layers.Conv2D(base, 3, padding="same", activation="relu")(inputs)
    x1 = layers.Conv2D(base, 3, padding="same", activation="relu")(x1)
    p1 = layers.MaxPooling2D()(x1)

    # Encoder 2
    x2 = layers.Conv2D(base*2, 3, padding="same", activation="relu")(p1)
    x2 = layers.Conv2D(base*2, 3, padding="same", activation="relu")(x2)
    p2 = layers.MaxPooling2D()(x2)

    # Bottleneck
    b  = layers.Conv2D(base*4, 3, padding="same", activation="relu")(p2)
    b  = layers.Conv2D(base*4, 3, padding="same", activation="relu")(b)

    # Decoder 2
    u2 = layers.UpSampling2D()(b)
    u2 = layers.Concatenate()([u2, x2])
    d2 = layers.Conv2D(base*2, 3, padding="same", activation="relu")(u2)
    d2 = layers.Conv2D(base*2, 3, padding="same", activation="relu")(d2)

    # Decoder 1
    u1 = layers.UpSampling2D()(d2)
    u1 = layers.Concatenate()([u1, x1])
    d1 = layers.Conv2D(base, 3, padding="same", activation="relu")(u1)
    d1 = layers.Conv2D(base, 3, padding="same", activation="relu")(d1)

    outputs = layers.Conv2D(1, 1, padding="same")(d1)
    return keras.Model(inputs, outputs, name="UNetMedium")

In [42]:
model_med = build_unet_medium(X_train_norm.shape[1:], base=48)

model_med.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.Huber(delta=0.05),
    metrics=[keras.metrics.MeanAbsoluteError(name="mae"),
             keras.metrics.RootMeanSquaredError(name="rmse")]
)

callbacks = [
    keras.callbacks.ModelCheckpoint("models/unet_modelA_Kt_huber_medium_best.keras",
                                    monitor="val_loss", save_best_only=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6,
                                     min_lr=1e-5, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=15,
                                  restore_best_weights=True, verbose=1)
]

hist = model_med.fit(
    X_train_norm, y_train_Kt,
    validation_data=(X_val_norm, y_val_Kt),
    epochs=250,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/250
19/20 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.0178 - mae: 0.3805 - rmse: 0.5881
Epoch 1: val_loss improved from None to 0.00468, saving model to models/unet_modelA_Kt_huber_medium_best.keras
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 60ms/step - loss: 0.0109 - mae: 0.2419 - rmse: 0.4127 - val_loss: 0.0047 - val_mae: 0.1158 - val_rmse: 0.1568 - learning_rate: 0.0010
Epoch 2/250
19/20 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.0053 - mae: 0.1291 - rmse: 0.1635
Epoch 2: val_loss did not improve from 0.00468
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - loss: 0.0051 - mae: 0.1255 - rmse: 0.1577 - val_loss: 0.0073 - val_mae: 0.1708 - val_rmse: 0.2125 - learning_rate: 0.0010
Epoch 3/250
19/20 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.0075 - mae: 0.1734 - rmse: 0.2132
Epoch 3: val_loss improved from 0.00468 to 0.00405, saving model to models/unet_modelA_Kt_huber_medium_best.keras
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - loss: 0.0059 - mae: 0.1421 - rmse: 0.1795 - val_loss: 0.0040 - val_mae

In [43]:
import numpy as np
import pandas as pd
from tensorflow import keras
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

best_Kt = keras.models.load_model("models/unet_modelA_Kt_huber_medium_best.keras")
print("✅ Loaded:", "models/unet_modelA_Kt_huber_medium_best.keras")

# Predict Kt on val
Kt_pred = best_Kt.predict(X_val_norm, verbose=0).astype("float32")   # (Nval,H,W,1)
Kt_true = y_val_Kt.astype("float32")

# Get matching I0_maps for val subset
I0_val = I0_maps[val_idx].astype("float32")  # (Nval,H,W)

# Convert to SIS
SIS_pred = (Kt_pred[..., 0] * I0_val).astype("float32")
SIS_true = (y_A[val_idx][..., 0]).astype("float32")  # original SIS target

# Pixel-wise metrics (all pixels pooled)
SIS_true_flat = SIS_true.reshape(-1)
SIS_pred_flat = SIS_pred.reshape(-1)

pixel_mae  = mean_absolute_error(SIS_true_flat, SIS_pred_flat)
pixel_rmse = np.sqrt(mean_squared_error(SIS_true_flat, SIS_pred_flat))
pixel_r2   = r2_score(SIS_true_flat, SIS_pred_flat)

# City-mean per day metrics
true_mean = SIS_true.mean(axis=(1,2))
pred_mean = SIS_pred.mean(axis=(1,2))

city_mae  = mean_absolute_error(true_mean, pred_mean)
city_rmse = np.sqrt(mean_squared_error(true_mean, pred_mean))
city_r2   = r2_score(true_mean, pred_mean)
city_bias = float((pred_mean - true_mean).mean())

print("\nValidation metrics (SIS W/m²):")
print(f"Pixel-wise  MAE : {pixel_mae:.3f}")
print(f"Pixel-wise  RMSE: {pixel_rmse:.3f}")
print(f"Pixel-wise  R²  : {pixel_r2:.4f}")
print(f"City-mean   MAE : {city_mae:.3f}")
print(f"City-mean   RMSE: {city_rmse:.3f}")
print(f"City-mean   R²  : {city_r2:.4f}")
print(f"City-mean   bias: {city_bias:.3f}")

# Monthly bias/MAE table (val months)
val_times = pd.to_datetime(kept_times[val_idx])
dfm = pd.DataFrame({
    "time": val_times,
    "month": val_times.strftime("%Y-%m"),
    "true_mean": true_mean,
    "pred_mean": pred_mean
})
dfm["bias"] = dfm["pred_mean"] - dfm["true_mean"]
dfm["abs_err"] = np.abs(dfm["bias"])

monthly = dfm.groupby("month").agg(
    days=("time", "count"),
    mean_true=("true_mean", "mean"),
    mean_pred=("pred_mean", "mean"),
    bias=("bias", "mean"),
    mae=("abs_err", "mean")
).reset_index()

print("\nMonthly city-mean stats (validation):")
print(monthly)

# Worst 5 days by abs error
worst5 = dfm.sort_values("abs_err", ascending=False).head(5)[["time","true_mean","pred_mean","bias","abs_err"]]
print("\nWorst 5 validation days (city-mean abs error):")
print(worst5)


✅ Loaded: models/unet_modelA_Kt_huber_medium_best.keras

Validation metrics (SIS W/m²):
Pixel-wise  MAE : 7.667
Pixel-wise  RMSE: 10.603
Pixel-wise  R²  : 0.6419
City-mean   MAE : 6.152
City-mean   RMSE: 8.743
City-mean   R²  : 0.7357
City-mean   bias: -1.646

Monthly city-mean stats (validation):
     month  days  mean_true  mean_pred      bias       mae
0  2024-11    30  35.203251  33.122372 -2.080875  8.401525
1  2024-12    31  18.482178  17.256186 -1.225992  3.974768

Worst 5 validation days (city-mean abs error):
         time  true_mean  pred_mean       bias    abs_err
0  2024-11-01  18.807501  42.044846  23.237345  23.237345
4  2024-11-05  83.690002  62.382004 -21.307999  21.307999
49 2024-12-20  37.087502  15.935825 -21.151676  21.151676
23 2024-11-24  38.597500  20.183849 -18.413651  18.413651
17 2024-11-18  46.207500  28.679970 -17.527531  17.527531


**Observation**

Clear improvement across the main accuracy metrics compared to UNetSmall + Huber:

- Pixel-wise: MAE 7.667 W/m² (↓ from 8.109), RMSE 10.603 (↓), R² 0.6419 (↑)
- City-mean:  MAE 6.152 W/m² (↓ from 6.407), RMSE 8.743 (↓), R² 0.7357 (↑)

Calibration note:
- City-mean bias remains negative: −1.646 W/m² (similar to −1.619) → UNetMedium improves accuracy but it does not significantly change the overall bias.

Monthly behaviour:
- November bias = −2.081 W/m²
- December bias = −1.226 W/m²
The model still underpredicts on average in both months but December is less negative (closer to zero).

Worst-day errors remain but the high-SIS underprediction improves:
- 2024-11-05 improves from −24.59 (UNetSmall) → −21.31 W/m² (UNetMedium)
The largest overprediction day is still:
- 2024-11-01: +23.24 W/m²

**Decision:**
UNetMedium + Huber is the current best Model A configuration: improved accuracy and higher R², while worst-case days are reduced but not eliminated.


## 12.15 Freeze Model A (best baseline) + save normalization stats

Now that UNetMedium + Huber + Kt → SIS is confirmed as the best-performing Model A configuration, we freeze it as our reference baseline. In this step we:

- Define the final best-model path for Model A
- Save the train-only normalization stats (train_mean, train_std) so future inference uses the exact same scaling
- Save a small JSON summary of the final validation metrics for reporting/reproducibility

In [44]:
import os
import json
import numpy as np

# Final best model path 
BEST_MODEL_A_PATH = "models/unet_modelA_Kt_huber_medium_best.keras"

# Ensure models folder exists
os.makedirs("models", exist_ok=True)

# Save normalization stats (train-only) used for this model
np.save("models/unetA_train_mean.npy", train_mean)
np.save("models/unetA_train_std.npy", train_std)

print("✅ Final Model A saved:", BEST_MODEL_A_PATH)
print("✅ Normalization stats saved:")
print(" - models/unetA_train_mean.npy")
print(" - models/unetA_train_std.npy")

# Save final evaluation metrics 
metrics_path = "models/unet_modelA_final_metrics.json"

final_metrics = {}
if "pixel_mae" in globals(): final_metrics["pixel_mae_Wm2"] = float(pixel_mae)
if "pixel_rmse" in globals(): final_metrics["pixel_rmse_Wm2"] = float(pixel_rmse)
if "pixel_r2" in globals(): final_metrics["pixel_r2"] = float(pixel_r2)

if "city_mae" in globals(): final_metrics["city_mae_Wm2"] = float(city_mae)
if "city_rmse" in globals(): final_metrics["city_rmse_Wm2"] = float(city_rmse)
if "city_r2" in globals(): final_metrics["city_r2"] = float(city_r2)
if "city_bias" in globals(): final_metrics["city_bias_Wm2"] = float(city_bias)

if len(final_metrics) > 0:
    with open(metrics_path, "w") as f:
        json.dump(final_metrics, f, indent=2)
    print("✅ Final metrics saved:", metrics_path)
else:
    print("ℹ️ Metrics variables not found in globals(). Skipping metrics JSON save.")


✅ Final Model A saved: models/unet_modelA_Kt_huber_medium_best.keras
✅ Normalization stats saved:
 - models/unetA_train_mean.npy
 - models/unetA_train_std.npy
✅ Final metrics saved: models/unet_modelA_final_metrics.json


## 13. Model B — Future / Seasonal-estimate U-Net (2025+ compatible)
**Goal**

Train a “future-safe” U-Net style CNN that can predict daily radiation maps when some feature sources are missing (e.g., no Sentinel-2 in 2025 and tcc missing in CAMS NRT).

**Key idea**

- Model A (2024) used the richest feature stack (CAMS + DEM + Sentinel-2 + seasonality).
- Model B (2025+) must use only channels that exist reliably across 2024–2025 in the fused dataset.
- We keep the physically-informed target: predict clearness index Kt then reconstruct into SIS.
  
**Inputs**
- CAMS (stable): aod550, tcwv
- DEM (static): elevation, slope, aspect
- Seasonality: doy_sin, doy_cos (broadcast as 20×20 constant maps)

**Output**
- Predicted Kt map: (20, 20, 1)
- Convert to SIS (W/m²) for evaluation and UI.

### 13.1 Define helper functions + Model B feature list

In [45]:
import numpy as np
import pandas as pd

UNET_FEATURES_B = [
    # CAMS 
    "aod550", "tcwv",

    # DEM 
    "elevation", "slope", "aspect",

    # Seasonality channels 
    "doy_sin", "doy_cos",
]

UNET_TARGET = "SIS"  # SIS stored in fused; convert SIS -> Kt later

def assert_vars_exist(ds, vars_list):
    """Raise a clear error if any required variables are missing in an xarray Dataset."""
    missing = [v for v in vars_list if v not in ds.data_vars and v not in ds.coords]
    if missing:
        raise ValueError(f"Missing variables in dataset: {missing}")

def nan_ratio_total(arr: np.ndarray) -> float:
    """NaN ratio for a numpy array."""
    return float(np.isnan(arr).sum()) / arr.size

print("✅ Model B feature channels:", len(UNET_FEATURES_B))
print("Features:", UNET_FEATURES_B)


✅ Model B feature channels: 7
Features: ['aod550', 'tcwv', 'elevation', 'slope', 'aspect', 'doy_sin', 'doy_cos']


### 13.2 Build training tensors for Model B 
In this step, we will:
- Uses fused (already aligned on SARAH grid).
- Builds X_B with shape (N_days, 20, 20, 7)
- Builds y_SIS_B (raw SIS) and also prepares I0_maps_B (needed later for Kt and SIS reconstruction)
- Adds doy_sin/doy_cos as 1D over time then broadcasts to 20×20 maps per day
- Fills NaNs in feature maps using day spatial mean, fallback to global mean (computed from training period)

In [48]:
import numpy as np
import pandas as pd

assert "fused" in globals(), "Run fusion steps first: `fused` not found."
assert_vars_exist(fused, ["SIS"] + [v for v in UNET_FEATURES_B if v not in ["doy_sin", "doy_cos"]])

fused_B = fused.copy()

# Add seasonality channels (1D over time)
times = pd.to_datetime(fused_B["time"].values)
doy = pd.Series(times).dt.dayofyear.values.astype(int)

fused_B = fused_B.assign(
    doy_sin=("time", np.sin(2 * np.pi * doy / 365.25).astype("float32")),
    doy_cos=("time", np.cos(2 * np.pi * doy / 365.25).astype("float32")),
)

H = fused_B.sizes["lat"]
W = fused_B.sizes["lon"]
C = len(UNET_FEATURES_B)

print("Grid:", H, "x", W, "| Channels:", C, "| Days:", fused_B.sizes["time"])
assert H == 20 and W == 20, "Expected SARAH Eindhoven grid to be 20×20."

# Compute global fallback means (training period only: 2024)
global_means_B = {}
fused_B_train_for_means = fused_B.sel(time=slice("2024-01-01", "2024-12-31"))

for feat in UNET_FEATURES_B:
    if feat in ["doy_sin", "doy_cos"]:
        continue
    vals = fused_B_train_for_means[feat].values.astype("float32")  # (time, lat, lon)
    global_means_B[feat] = float(np.nanmean(vals))

print("✅ Global feature means (Model B, from 2024 only) computed.")

# Build tensors (X_B, y_SIS_B)
X_list, y_list, kept_times_B = [], [], []

filled_cells_total = 0
skipped_y_nan = 0

for t in fused_B["time"].values:
    day = fused_B.sel(time=t)

    # Target SIS map
    y_day = day[UNET_TARGET].values.astype("float32")  # (H,W)
    if np.isnan(y_day).any():
        skipped_y_nan += 1
        continue

    # Broadcast seasonality maps
    sin_map = np.full((H, W), float(day["doy_sin"].values), dtype="float32")
    cos_map = np.full((H, W), float(day["doy_cos"].values), dtype="float32")

    channels = []
    for feat in UNET_FEATURES_B:
        if feat == "doy_sin":
            ch = sin_map
        elif feat == "doy_cos":
            ch = cos_map
        else:
            ch = day[feat].values.astype("float32")

            # Fill NaNs
            if np.isnan(ch).any():
                nan_mask = np.isnan(ch)
                filled_cells_total += int(nan_mask.sum())

                day_mean = float(np.nanmean(ch)) if np.isfinite(np.nanmean(ch)) else np.nan
                fill_value = day_mean if np.isfinite(day_mean) else global_means_B[feat]

                ch = ch.copy()
                ch[nan_mask] = fill_value

        channels.append(ch)

    X_day = np.stack(channels, axis=-1)  # (H,W,C)

    # Safety check
    if np.isnan(X_day).any():
        continue

    X_list.append(X_day)
    y_list.append(y_day)
    kept_times_B.append(pd.to_datetime(t))

X_B = np.stack(X_list, axis=0).astype("float32")            # (N,H,W,C)
y_SIS_B = np.stack(y_list, axis=0).astype("float32")[..., None]  # (N,H,W,1)
kept_times_B = pd.to_datetime(kept_times_B)

print("\n✅ Built Model B tensors:")
print("X_B:", X_B.shape, "| y_SIS_B:", y_SIS_B.shape)
print("Skipped days due to NaN in target SIS:", skipped_y_nan)
print("Total filled NaN cells across all feature maps:", filled_cells_total)
print("NaNs in X_B:", int(np.isnan(X_B).sum()), "| NaNs in y_SIS_B:", int(np.isnan(y_SIS_B).sum()))

# Prepare I0 maps 
def extraterrestrial_radiation_Wm2(lat_deg, doy):
    lat = np.deg2rad(lat_deg.astype(float))
    doy = doy.astype(float)

    dr = 1 + 0.033 * np.cos(2 * np.pi * doy / 365.0)
    delta = 0.409 * np.sin(2 * np.pi * doy / 365.0 - 1.39)
    ws = np.arccos(np.clip(-np.tan(lat) * np.tan(delta), -1, 1))

    Gsc = 1367.0
    Ra = (24 * 3600 / np.pi) * Gsc * dr * (
        ws * np.sin(lat) * np.sin(delta) +
        np.cos(lat) * np.cos(delta) * np.sin(ws)
    )
    return Ra / (24 * 3600)

# lat map (H,W)
lat_vec = fused_B["lat"].values.astype("float32")
lat_map = np.repeat(lat_vec[:, None], W, axis=1).astype("float32")  # (H,W)

doy_kept = pd.to_datetime(kept_times_B).dayofyear.values.astype(int)

I0_maps_B = []
for d in doy_kept:
    I0_day = extraterrestrial_radiation_Wm2(lat_map, np.full_like(lat_map, d))
    I0_maps_B.append(I0_day.astype("float32"))

I0_maps_B = np.stack(I0_maps_B, axis=0).astype("float32")  # (N,H,W)

print("\n✅ Built I0_maps_B:", I0_maps_B.shape,
      "| range:", float(I0_maps_B.min()), "→", float(I0_maps_B.max()))


Grid: 20 x 20 | Channels: 7 | Days: 457
✅ Global feature means (Model B, from 2024 only) computed.

✅ Built Model B tensors:
X_B: (457, 20, 20, 7) | y_SIS_B: (457, 20, 20, 1)
Skipped days due to NaN in target SIS: 0
Total filled NaN cells across all feature maps: 800
NaNs in X_B: 0 | NaNs in y_SIS_B: 0

✅ Built I0_maps_B: (457, 20, 20) | range: 72.9541244506836 → 483.22625732421875


C:\Users\Student\AppData\Local\Temp\ipykernel_6028\2000153232.py:70: RuntimeWarning: Mean of empty slice
  day_mean = float(np.nanmean(ch)) if np.isfinite(np.nanmean(ch)) else np.nan


### 13.3 Build Kt targets (Model B) + time-based split (2024 → 2025 Q1)

In [49]:
import numpy as np
import pandas as pd

# Check
assert "X_B" in globals() and "y_SIS_B" in globals(), "Run 13.2 first (X_B / y_SIS_B missing)."
assert "I0_maps_B" in globals(), "Run 13.2 first (I0_maps_B missing)."
assert "kept_times_B" in globals(), "Need kept_times_B from 13.2."

kept_times_B = pd.to_datetime(kept_times_B)

# Build Kt targets from SIS
eps = 1e-6
SIS_B = y_SIS_B[..., 0].astype("float32")  # (N,H,W)
I0_B  = I0_maps_B.astype("float32")        # (N,H,W)

Kt_B = SIS_B / (I0_B + eps)
Kt_B = np.clip(Kt_B, 0.0, 1.2).astype("float32")  # physical-ish bounds

y_Kt_B = Kt_B[..., None]  # (N,H,W,1)

print("✅ Built y_Kt_B:", y_Kt_B.shape, "| range:", float(y_Kt_B.min()), "→", float(y_Kt_B.max()))
print("NaNs in y_Kt_B:", int(np.isnan(y_Kt_B).sum()))

# Time-based split: train on 2024, validate on 2025 Q1
train_idx_B = (kept_times_B >= "2024-01-01") & (kept_times_B <= "2024-12-31")
val_idx_B   = (kept_times_B >= "2025-01-01") & (kept_times_B <= "2025-03-31")

X_train_B, y_train_Kt_B = X_B[train_idx_B].astype("float32"), y_Kt_B[train_idx_B].astype("float32")
X_val_B,   y_val_Kt_B   = X_B[val_idx_B].astype("float32"),   y_Kt_B[val_idx_B].astype("float32")

print("\nModel B split (Kt target):")
print("Train:", X_train_B.shape, y_train_Kt_B.shape, "| days:", int(train_idx_B.sum()))
print("Val  :", X_val_B.shape,   y_val_Kt_B.shape,   "| days:", int(val_idx_B.sum()))

print("Train range:", kept_times_B[train_idx_B].min(), "→", kept_times_B[train_idx_B].max())
print("Val range  :", kept_times_B[val_idx_B].min(),   "→", kept_times_B[val_idx_B].max())

# Keep matching I0 for validation reconstruction later
I0_train_B = I0_maps_B[train_idx_B].astype("float32")
I0_val_B   = I0_maps_B[val_idx_B].astype("float32")

print("\n✅ I0 aligned:")
print("I0_train_B:", I0_train_B.shape, "| I0_val_B:", I0_val_B.shape)

# Final sanity: no NaNs
print("\nNaN checks:")
print("NaNs X_train_B:", int(np.isnan(X_train_B).sum()))
print("NaNs X_val_B  :", int(np.isnan(X_val_B).sum()))
print("NaNs y_train_Kt_B:", int(np.isnan(y_train_Kt_B).sum()))
print("NaNs y_val_Kt_B  :", int(np.isnan(y_val_Kt_B).sum()))
print("NaNs I0_val_B:", int(np.isnan(I0_val_B).sum()))


✅ Built y_Kt_B: (457, 20, 20, 1) | range: 0.06553531438112259 → 0.757155179977417
NaNs in y_Kt_B: 0

Model B split (Kt target):
Train: (366, 20, 20, 7) (366, 20, 20, 1) | days: 366
Val  : (90, 20, 20, 7) (90, 20, 20, 1) | days: 90
Train range: 2024-01-01 00:00:00 → 2024-12-31 00:00:00
Val range  : 2025-01-01 00:00:00 → 2025-03-31 00:00:00

✅ I0 aligned:
I0_train_B: (366, 20, 20) | I0_val_B: (90, 20, 20)

NaN checks:
NaNs X_train_B: 0
NaNs X_val_B  : 0
NaNs y_train_Kt_B: 0
NaNs y_val_Kt_B  : 0
NaNs I0_val_B: 0


### 13.4 Normalize inputs (train-only stats) + build TensorFlow datasets

In [50]:
import numpy as np
import tensorflow as tf

assert "X_train_B" in globals() and "X_val_B" in globals(), "Run 13.3 first (X_train_B / X_val_B missing)."
assert "y_train_Kt_B" in globals() and "y_val_Kt_B" in globals(), "Run 13.3 first (y_train_Kt_B / y_val_Kt_B missing)."

eps = 1e-6

# Compute per-channel mean/std from TRAIN only
# X_train_B shape: (N, H, W, C)
train_mean_B = X_train_B.mean(axis=(0, 1, 2), keepdims=True).astype("float32")  # (1,1,1,C)
train_std_B  = X_train_B.std(axis=(0, 1, 2), keepdims=True).astype("float32")   # (1,1,1,C)
train_std_B  = np.maximum(train_std_B, eps)

print("train_mean_B shape:", train_mean_B.shape, "| train_std_B shape:", train_std_B.shape)
print("train_std_B min/max:", float(train_std_B.min()), "→", float(train_std_B.max()))

# Apply normalization
X_train_B_norm = ((X_train_B - train_mean_B) / train_std_B).astype("float32")
X_val_B_norm   = ((X_val_B   - train_mean_B) / train_std_B).astype("float32")

# Targets float32
y_train_B_f = y_train_Kt_B.astype("float32")
y_val_B_f   = y_val_Kt_B.astype("float32")

print("\n✅ Normalized tensors (Model B):")
print("X_train_B_norm:", X_train_B_norm.shape, "| y_train_B:", y_train_B_f.shape)
print("X_val_B_norm  :", X_val_B_norm.shape,   "| y_val_B  :", y_val_B_f.shape)

# NaN checks
print("\nNaN checks:")
print("NaNs in X_train_B_norm:", int(np.isnan(X_train_B_norm).sum()))
print("NaNs in X_val_B_norm  :", int(np.isnan(X_val_B_norm).sum()))
print("NaNs in y_train_B_f   :", int(np.isnan(y_train_B_f).sum()))
print("NaNs in y_val_B_f     :", int(np.isnan(y_val_B_f).sum()))

# Build tf.data datasets
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

ds_train_B = tf.data.Dataset.from_tensor_slices((X_train_B_norm, y_train_B_f))
ds_train_B = ds_train_B.shuffle(buffer_size=len(X_train_B_norm), reshuffle_each_iteration=True)
ds_train_B = ds_train_B.batch(BATCH_SIZE).prefetch(AUTOTUNE)

ds_val_B = tf.data.Dataset.from_tensor_slices((X_val_B_norm, y_val_B_f))
ds_val_B = ds_val_B.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# Quick batch shape check
for xb, yb in ds_train_B.take(1):
    print("\nBatch shapes:")
    print("X:", xb.shape, "y:", yb.shape)

print("\n✅ tf.data datasets ready (Model B).")


train_mean_B shape: (1, 1, 1, 7) | train_std_B shape: (1, 1, 1, 7)
train_std_B min/max: 0.0929357185959816 → 104.29857635498047

✅ Normalized tensors (Model B):
X_train_B_norm: (366, 20, 20, 7) | y_train_B: (366, 20, 20, 1)
X_val_B_norm  : (90, 20, 20, 7) | y_val_B  : (90, 20, 20, 1)

NaN checks:
NaNs in X_train_B_norm: 0
NaNs in X_val_B_norm  : 0
NaNs in y_train_B_f   : 0
NaNs in y_val_B_f     : 0

Batch shapes:
X: (16, 20, 20, 7) y: (16, 20, 20, 1)

✅ tf.data datasets ready (Model B).


### 13.5 Build Model B U-Net (UNetMedium) + compile (Huber on Kt)

In [51]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

assert "X_train_B_norm" in globals(), "Run 13.4 first (X_train_B_norm missing)."

H, W, C = X_train_B_norm.shape[1], X_train_B_norm.shape[2], X_train_B_norm.shape[3]
print("Model B input shape:", (H, W, C))

def build_unet_medium(input_shape, base=48):
    inputs = keras.Input(shape=input_shape, name="input")

    # Encoder 1
    x1 = layers.Conv2D(base, 3, padding="same", activation="relu")(inputs)
    x1 = layers.Conv2D(base, 3, padding="same", activation="relu")(x1)
    p1 = layers.MaxPooling2D()(x1)     # 20 -> 10

    # Encoder 2
    x2 = layers.Conv2D(base * 2, 3, padding="same", activation="relu")(p1)
    x2 = layers.Conv2D(base * 2, 3, padding="same", activation="relu")(x2)
    p2 = layers.MaxPooling2D()(x2)     # 10 -> 5

    # Bottleneck
    b  = layers.Conv2D(base * 4, 3, padding="same", activation="relu")(p2)
    b  = layers.Conv2D(base * 4, 3, padding="same", activation="relu")(b)

    # Decoder 2
    u2 = layers.UpSampling2D()(b)      # 5 -> 10
    u2 = layers.Concatenate()([u2, x2])
    d2 = layers.Conv2D(base * 2, 3, padding="same", activation="relu")(u2)
    d2 = layers.Conv2D(base * 2, 3, padding="same", activation="relu")(d2)

    # Decoder 1
    u1 = layers.UpSampling2D()(d2)     # 10 -> 20
    u1 = layers.Concatenate()([u1, x1])
    d1 = layers.Conv2D(base, 3, padding="same", activation="relu")(u1)
    d1 = layers.Conv2D(base, 3, padding="same", activation="relu")(d1)

    # Output: Kt map
    outputs = layers.Conv2D(1, 1, padding="same", activation="linear", name="kt_pred")(d1)

    return keras.Model(inputs, outputs, name="UNetMedium_B")

model_B = build_unet_medium(input_shape=(H, W, C), base=48)
model_B.summary()

# Compile (Huber on Kt)
model_B.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.Huber(delta=0.05),   # delta in Kt units
    metrics=[
        keras.metrics.MeanAbsoluteError(name="mae"),
        keras.metrics.RootMeanSquaredError(name="rmse"),
    ],
)

print("✅ Model B compiled (UNetMedium + Huber on Kt).")

Model B input shape: (20, 20, 7)


Model: "UNetMedium_B"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)            │ (None, 20, 20, 7)         │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_11 (Conv2D)            │ (None, 20, 20, 48)        │           3,072 │ input[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_12 (Conv2D)            │ (None, 20, 20, 48)        │          20,784 │ conv2d_11[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_2               │ (None, 10, 10, 48)        │               0 │ conv2d_12[0][0]            │
│ (MaxPooling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_13 (Conv2D)            │ (None, 10, 10, 96)        │          41,568 │ max_pooling2d_2[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_14 (Conv2D)            │ (None, 10, 10, 96)        │          83,040 │ conv2d_13[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_3               │ (None, 5, 5, 96)          │               0 │ conv2d_14[0][0]            │
│ (MaxPooling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_15 (Conv2D)            │ (None, 5, 5, 192)         │         166,080 │ max_pooling2d_3[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_16 (Conv2D)            │ (None, 5, 5, 192)         │         331,968 │ conv2d_15[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ up_sampling2d_2               │ (None, 10, 10, 192)       │               0 │ conv2d_16[0][0]            │
│ (UpSampling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concatenate_2 (Concatenate)   │ (None, 10, 10, 288)       │               0 │ up_sampling2d_2[0][0],     │
│                               │                           │                 │ conv2d_14[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_17 (Conv2D)            │ (None, 10, 10, 96)        │         248,928 │ concatenate_2[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_18 (Conv2D)            │ (None, 10, 10, 96)        │          83,040 │ conv2d_17[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ up_sampling2d_3               │ (None, 20, 20, 96)        │               0 │ conv2d_18[0][0]            │
│ (UpSampling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concatenate_3 (Concatenate)   │ (None, 20, 20, 144)       │               

 Total params: 1,061,569 (4.05 MB)

 Trainable params: 1,061,569 (4.05 MB)

 Non-trainable params: 0 (0.00 B)

✅ Model B compiled (UNetMedium + Huber on Kt).


### 13.6 Train Model B (UNetMedium + Huber on Kt)

- Train: 2024 (366 days)
- Val: Jan–Mar 2025 (90 days)

We save the best model + early stop + reduce LR.

In [52]:
from pathlib import Path
from tensorflow import keras
import tensorflow as tf
import numpy as np

assert "model_B" in globals(), "Run 13.5 first (model_B missing)."
assert "X_train_B_norm" in globals() and "X_val_B_norm" in globals(), "Run 13.4 first (normalized X missing)."
assert "y_train_Kt_B" in globals() and "y_val_Kt_B" in globals(), "Need y_train_Kt_B / y_val_Kt_B from 13.2/13.3."

# Ensure float32
X_train = X_train_B_norm.astype("float32")
X_val   = X_val_B_norm.astype("float32")
y_train = y_train_Kt_B.astype("float32")
y_val   = y_val_Kt_B.astype("float32")

print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

BEST_MODEL_B_PATH = MODEL_DIR / "unet_modelB_Kt_huber_medium_best.keras"

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_B_PATH),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=6,
        min_lr=1e-5,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
]

history_B = model_B.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=250,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Training finished (Model B).")
print("Best model path:", BEST_MODEL_B_PATH)

Train: (366, 20, 20, 7) (366, 20, 20, 1)
Val  : (90, 20, 20, 7) (90, 20, 20, 1)
Epoch 1/250
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 0.0125 - mae: 0.2740 - rmse: 0.3725
Epoch 1: val_loss improved from None to 0.00798, saving model to models\unet_modelB_Kt_huber_medium_best.keras
23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 63ms/step - loss: 0.0087 - mae: 0.1967 - rmse: 0.2776 - val_loss: 0.0080 - val_mae: 0.1832 - val_rmse: 0.2213 - learning_rate: 0.0010
Epoch 2/250
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 0.0054 - mae: 0.1301 - rmse: 0.1627
Epoch 2: val_loss improved from 0.00798 to 0.00695, saving model to models\unet_modelB_Kt_huber_medium_best.keras
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - loss: 0.0054 - mae: 0.1312 - rmse: 0.1628 - val_loss: 0.0069 - val_mae: 0.1628 - val_rmse: 0.1876 - learning_rate: 0.0010
Epoch 3/250
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 0.0049 - mae: 0.1208 - rmse: 0.1524
Epoch 3: val_loss improved from 0.00695 to 0.00656, saving model to models\unet

### 13.7 Evaluate Model B in SIS units (pixel-wise + city-mean + monthly + worst days)

In [53]:
import numpy as np
import pandas as pd
from tensorflow import keras
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

assert "X_val_B_norm" in globals(), "Need X_val_B_norm (run 13.4)."
assert "y_val_Kt_B" in globals(), "Need y_val_Kt_B (run 13.3)."
assert "y_SIS_B" in globals(), "Need y_SIS_B from 13.2 (SIS tensor)."
assert "I0_val_B" in globals(), "Need I0_val_B from 13.3."
assert "kept_times_B" in globals() and "val_idx_B" in globals(), "Need kept_times_B / val_idx_B from 13.3."
assert "BEST_MODEL_B_PATH" in globals() or True, "Best path printed in training cell."

BEST_MODEL_B_PATH = "models/unet_modelB_Kt_huber_medium_best.keras"

best_B = keras.models.load_model(BEST_MODEL_B_PATH)
print("✅ Loaded Model B:", BEST_MODEL_B_PATH)

# Predict Kt on validation
Kt_pred = best_B.predict(X_val_B_norm, verbose=0).astype("float32")   # (Nval,H,W,1)
Kt_true = y_val_Kt_B.astype("float32")

# Convert to SIS using I0
SIS_pred = (Kt_pred[..., 0] * I0_val_B).astype("float32")            # (Nval,H,W)
SIS_true = (y_SIS_B[val_idx_B][..., 0]).astype("float32")            # (Nval,H,W)

# Pixel-wise metrics 
SIS_true_flat = SIS_true.reshape(-1)
SIS_pred_flat = SIS_pred.reshape(-1)

pixel_mae  = mean_absolute_error(SIS_true_flat, SIS_pred_flat)
pixel_rmse = np.sqrt(mean_squared_error(SIS_true_flat, SIS_pred_flat))
pixel_r2   = r2_score(SIS_true_flat, SIS_pred_flat)

# City-mean per day metrics
true_mean = SIS_true.mean(axis=(1, 2))
pred_mean = SIS_pred.mean(axis=(1, 2))

city_mae  = mean_absolute_error(true_mean, pred_mean)
city_rmse = np.sqrt(mean_squared_error(true_mean, pred_mean))
city_r2   = r2_score(true_mean, pred_mean)
city_bias = float((pred_mean - true_mean).mean())

print("\nValidation metrics (Model B → SIS W/m²):")
print(f"Pixel-wise  MAE : {pixel_mae:.3f}")
print(f"Pixel-wise  RMSE: {pixel_rmse:.3f}")
print(f"Pixel-wise  R²  : {pixel_r2:.4f}")
print(f"City-mean   MAE : {city_mae:.3f}")
print(f"City-mean   RMSE: {city_rmse:.3f}")
print(f"City-mean   R²  : {city_r2:.4f}")
print(f"City-mean   bias: {city_bias:.3f}")

# Monthly table + worst days
val_times = pd.to_datetime(kept_times_B[val_idx_B])  # should match X_val_B_norm order
assert len(val_times) == len(true_mean), "Mismatch: val_times vs validation arrays."

dfm = pd.DataFrame({
    "time": val_times,
    "month": val_times.strftime("%Y-%m"),
    "true_mean": true_mean,
    "pred_mean": pred_mean
})
dfm["bias"] = dfm["pred_mean"] - dfm["true_mean"]
dfm["abs_err"] = np.abs(dfm["bias"])

monthly = dfm.groupby("month").agg(
    days=("time", "count"),
    mean_true=("true_mean", "mean"),
    mean_pred=("pred_mean", "mean"),
    bias=("bias", "mean"),
    mae=("abs_err", "mean")
).reset_index()

print("\nMonthly city-mean stats (validation):")
print(monthly)

worst5 = dfm.sort_values("abs_err", ascending=False).head(5)[
    ["time", "true_mean", "pred_mean", "bias", "abs_err"]
]
print("\nWorst 5 validation days (city-mean abs error):")
print(worst5)


✅ Loaded Model B: models/unet_modelB_Kt_huber_medium_best.keras

Validation metrics (Model B → SIS W/m²):
Pixel-wise  MAE : 23.341
Pixel-wise  RMSE: 30.210
Pixel-wise  R²  : 0.7399
City-mean   MAE : 21.727
City-mean   RMSE: 28.350
City-mean   R²  : 0.7647
City-mean   bias: -2.510

Monthly city-mean stats (validation):
     month  days   mean_true   mean_pred       bias        mae
0  2025-01    31   27.186531   29.128063   1.941532  10.582226
1  2025-02    28   64.086159   65.147018   1.060860  20.696733
2  2025-03    31  141.366776  131.178680 -10.188087  33.802914

Worst 5 validation days (city-mean abs error):
         time   true_mean   pred_mean       bias    abs_err
69 2025-03-11   47.082500  132.392426  85.309921  85.309921
84 2025-03-26   66.042503  136.747757  70.705254  70.705254
78 2025-03-20  191.675003  127.759857 -63.915146  63.915146
79 2025-03-21  176.487503  112.826347 -63.661156  63.661156
68 2025-03-10  161.927505  102.218201 -59.709305  59.709305


**Observation**

Even without Sentinel-2 surface features and without tcc during the NRT period, the model still generalizes well:
- Pixel-wise: MAE 23.34 W/m², RMSE 30.21 W/m², R² 0.7399
- City-mean: MAE 21.73 W/m², RMSE 28.35 W/m², R² 0.7647
- Bias: −2.51 W/m² → slight overall underprediction, but relatively small compared to daily variation.

Monthly behaviour shows the limitation clearly:

- Jan & Feb: small overprediction on average (+1–2 W/m²)
- March: strong underprediction (−10.19 W/m²) + higher MAE
→ likely because March has more rapid atmospheric transitions (spring variability) and Model B has no high-resolution surface context.

Worst-day errors remain the main weakness:
- Largest error day is an extreme overprediction: 2025-03-11: +85.31 W/m²
- Several large underprediction days exist during high-SIS conditions: 2025-03-20/21: around −64 W/m²

**Conclusion**

Model B is accepted as the best “future-safe” model: it achieves good generalization (R² ≈ 0.76) using only stable, always-available features.
The remaining errors are concentrated on a few extreme days, which are hard to solve without missing channels (cloud structure, higher-frequency atmospheric signals or surface reflectance from Sentinel-2). Further architecture scaling is unlikely to fix this reliably without risking overfitting to 2024.

### 13.8 Freeze & save final Model B

In [55]:
import numpy as np
from pathlib import Path
import json

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

BEST_MODEL_B_PATH = "models/unet_modelB_Kt_huber_medium_best.keras"

# Save normalization stats
np.save(MODEL_DIR / "unetB_train_mean.npy", train_mean_B)
np.save(MODEL_DIR / "unetB_train_std.npy", train_std_B)

print("✅ Final Model B saved:", BEST_MODEL_B_PATH)
print("✅ Normalization stats saved:")
print(" -", MODEL_DIR / "unetB_train_mean.npy")
print(" -", MODEL_DIR / "unetB_train_std.npy")


modelB_metrics = {
    "model_name": "UNetMedium_B_Kt_Huber",
    "best_model_path": BEST_MODEL_B_PATH,
    "val_period": "2025-01-01 to 2025-03-31",
    "pixel_mae_Wm2": float(pixel_mae),
    "pixel_rmse_Wm2": float(pixel_rmse),
    "pixel_r2": float(pixel_r2),
    "city_mae_Wm2": float(city_mae),
    "city_rmse_Wm2": float(city_rmse),
    "city_r2": float(city_r2),
    "city_bias_Wm2": float(city_bias),
    "worst_days_citymean": worst5.assign(time=worst5["time"].astype(str)).to_dict(orient="records")
}

out_path = MODEL_DIR / "unet_modelB_final_metrics.json"
with open(out_path, "w") as f:
    json.dump(modelB_metrics, f, indent=2)

print("✅ Final metrics saved:", out_path)


✅ Final Model B saved: models/unet_modelB_Kt_huber_medium_best.keras
✅ Normalization stats saved:
 - models\unetB_train_mean.npy
 - models\unetB_train_std.npy
✅ Final metrics saved: models\unet_modelB_final_metrics.json


**Final Conclusion**

This notebook demonstrates the complete development, evaluation and selection of two complementary deep-learning models for daily solar radiation estimation using satellite and auxiliary data.

**What was achieved**

- A physically informed learning strategy was successfully implemented by predicting the clearness index (Kt) and reconstructing SIS (W/m²) using extraterrestrial radiation.
- Two models were designed for different operational contexts:
    - Model A (UNetMedium + Huber) for years with rich data availability (e.g. 2024), achieving high accuracy and strong spatial consistency.
    - Model B (UNetMedium + Huber) for future-safe estimation (2025+), trained only on stable, always-available features.

- A strict temporal split (train on 2024, validate on 2025) was applied, ensuring realistic generalization testing rather than random leakage.

- Extensive evaluation was performed at multiple levels: pixel-wise performance, city-mean daily performance, monthly behaviour and worst-day analysis.

Together, these steps confirm that the models do not merely memorize patterns, but learn meaningful physical–statistical relationships.

**Key limitations identified**

- Extreme days dominate the error budget, especially during rapid seasonal transitions (e.g. early spring).
- These errors are strongly linked to missing high-resolution cloud and surface information, such as Sentinel-2 reflectance, detailed cloud structure (tcc) or sub-daily atmospheric dynamics.
- Increasing model depth alone does not resolve these cases and risks overfitting to 2024-specific conditions.

These limitations are therefore data-driven, not architecture-driven.

**Model selection decision**

- Model A is finalized as the best-performance reference model when full data is available.
- Model B is accepted as the production-ready, future-safe model, offering robust performance using only reliable inputs.
- Both models are frozen, saved and documented with normalization statistics and evaluation metadata.

This concludes the modeling phase of the project and provides a stable foundation for demonstration, evaluation and future extension.